# TradeSense · Component 03 · Stage F · Notebook F2
### Train and compare order-book manipulation detectors (TGNN with graph edges, TGNN without, matched MLP, hand-made features + XGBoost) – validation only

**Aim.** Stage E2/E2d showed that plain scores see injected order-book manipulation only weakly. Here the detectors are *trained* on the labelled training injections (F1) and judged on validation days they never saw: the standard E1 set and the shifted F1 set (longer events, weaker and stronger magnitudes, and a new pattern, *flicker*, that was never in training).

| id | detector | trained on |
|---|---|---|
| M1 | TGNN, edges: rank chain + best-bid/best-ask + global | injected training days |
| M0 | the same TGNN without chain and spread edges (global links only) | same |
| B3d | matched MLP | same |
| B6 | 149 hand-made window features + XGBoost | same |
| B1-top5, B1-all, B5 | untrained scores from E2 | – |

**Reading rule (Protocol v3, `claude/protocol_v3_stageF.md`, written before any detector was trained).** Alarm thresholds are set per detector so that each raises the same number of false alarm episodes on clean time: 20, 100 and 400 per symbol-day (primary 100). H3a: M1 beats the best of B3d/B6 on the shifted set; H3b: M1 beats the best untrained score on the standard set; H3c: M1 beats the best untrained score on the held-out flicker pattern; H3d: M1 beats M0. A hypothesis holds only if the recall difference is at least 5 percentage points and the 95% event-bootstrap interval is above 0. Failed hypotheses are reported as failed.

**What it does not do.** No test day is opened. Everything is synthetic injection into real validation data. **Use a T4 GPU runtime**; about 2.5 to 3 hours in total. Training and scoring are resumable: if the time budget (`MAX_RUN_MINUTES`) ends, run all again.

## 1 · Settings (nothing to edit)

In [ ]:
import os, gc, sys, json, time, shutil, hashlib, platform, warnings
from pathlib import Path
import numpy as np, pandas as pd
warnings.filterwarnings('ignore')
from IPython.display import display
try:
    from google.colab import drive
    drive.mount('/content/drive')
except ImportError:
    print('Not in Colab -> using TRADESENSE_BASE.')
BASE = Path(os.environ.get('TRADESENSE_BASE', '/content/drive/MyDrive/research datasets')); assert BASE.is_dir(), f'Drive folder not found: {BASE}'
STAGEC_NAME = 'TGNN_StageC_GraphFeatures_v2'; D1_NAME = 'TGNN_StageD_v1'; E_NAME = 'TGNN_StageE_v1'; F_NAME = 'TGNN_StageF_v1'
SYMS = ['BTCUSDT', 'ETHUSDT']; TRAIN_DAYS = ['2024-05-01', '2024-06-01']; VAL_DAYS = ['2024-07-01']
CFG = {'hidden': 64, 'lr': 1e-3}; MAX_EPOCHS = 10; PATIENCE = 3; BATCH = 256; P_POS = 20000; NEG_MULT = 4; AMP = True
SEEDS = [301, 302, 303]; STRIDE = 3; BURDENS = [20, 100, 400]; PRIMARY = 100; N_BOOT = 2000; MIN_GAIN = 0.05
MAX_RUN_MINUTES = float(os.environ.get('MAX_RUN_MINUTES', '170'))
print('detector config', CFG, '| epochs<=', MAX_EPOCHS, 'patience', PATIENCE, 'batch', BATCH, 'positives/epoch', P_POS, 'neg x', NEG_MULT, '| seeds', SEEDS, '| scoring stride', STRIDE, '| burdens', BURDENS, '| max run minutes', MAX_RUN_MINUTES)

Mounted at /content/drive
detector config {'hidden': 64, 'lr': 0.001} | epochs<= 10 patience 3 batch 256 positives/epoch 20000 neg x 4 | seeds [301, 302, 303] | scoring stride 3 | burdens [20, 100, 400] | max run minutes 170.0


## 2 · Libraries (written to the Colab working folder, not to Drive). `staged_lib.py` is the exact D1 library; its hash is checked against D1.

In [ ]:
%%writefile staged_lib.py
"""TradeSense C03 - Stage D library (D1: loader, leak guards, baselines B1-B4, B3 MLP, resumable trainer).
Reads ONLY the frozen Stage C tensors. Never writes to Stage C. Test days are refused until a freeze file exists."""
import os, json, time, math, hashlib, shutil
from pathlib import Path
import numpy as np
import torch, torch.nn as nn

SEQ = 50; NN = 50; NCH = 6; GCH = 3; NT = 51; LOGQ = 2; GLS = 0; GIMB = 1
SYMS = ['BTCUSDT', 'ETHUSDT']
SPLITS = {'train': ['2024-05-01', '2024-06-01'], 'val': ['2024-07-01'], 'test': ['2024-08-01', '2024-09-01', '2024-10-01']}
def keys_of(split): return [f'{s}_{d}' for d in SPLITS[split] for s in SYMS]

# ---------------------------------------------------------------- small helpers
def sha256_file(p, chunk=16 << 20):
    h = hashlib.sha256()
    with open(p, 'rb') as f:
        while True:
            b = f.read(chunk)
            if not b: break
            h.update(b)
    return h.hexdigest()
def _j(o):
    if isinstance(o, (np.integer,)): return int(o)
    if isinstance(o, (np.floating,)): return float(o)
    if isinstance(o, np.ndarray): return o.tolist()
    if isinstance(o, Path): return str(o)
    raise TypeError(type(o))
def canonical_sha(obj): return hashlib.sha256(json.dumps(obj, sort_keys=True, default=_j).encode()).hexdigest()
def dump_json(obj, path):
    path = Path(path); path.parent.mkdir(parents=True, exist_ok=True); tmp = path.with_name(path.name + '.tmp')
    tmp.write_text(json.dumps(obj, indent=1, default=_j)); os.replace(tmp, path)
def put_json(obj, path):
    """write once; if a different file already exists it is NEVER replaced: the new content goes to <name>.rN.json"""
    path = Path(path)
    if path.exists():
        if json.loads(path.read_text()) == json.loads(json.dumps(obj, default=_j)): return path
        n = 1
        while path.with_name(f'{path.stem}.r{n}{path.suffix}').exists(): n += 1
        path = path.with_name(f'{path.stem}.r{n}{path.suffix}')
    dump_json(obj, path); return path
def check_frozen(path, cfg):
    """first call writes the frozen config; later calls must match exactly, otherwise resuming is refused"""
    path = Path(path)
    if not path.exists(): dump_json({'config': cfg, 'sha256': canonical_sha(cfg)}, path); return 'written'
    old = json.loads(path.read_text())
    if old['sha256'] != canonical_sha(cfg): raise RuntimeError(f'frozen config differs from the saved one ({path}); do not change settings or code while resuming a run')
    return 'matches'

# ---------------------------------------------------------------- the loader (all tensors in one block, optionally on the GPU)
class Bank:
    def __init__(self, stagec_dir, keys, device='cpu', freeze_file=None, chunk=100_000):
        self.dir = Path(stagec_dir) / 'sets'; self.keys = list(keys); self.device = device; self.summ = {}; total = 0
        for k in self.keys:
            s = json.loads((self.dir / k / 'set_summary.json').read_text()); self.summ[k] = s
            if s['split'] == 'test' and not (freeze_file and Path(freeze_file).exists()):
                raise PermissionError(f'{k} is a TEST day: refused because the freeze file does not exist yet')
            total += int(s['n_intervals'])
        self.nodes = torch.empty((total, NN, NCH), dtype=torch.float16, device=device); self.glob = torch.empty((total, GCH), dtype=torch.float16, device=device)
        self.off = {}; wins = []; sid = []; o = 0
        for i, k in enumerate(self.keys):
            arr = np.load(self.dir / k / 'nodes.npy', mmap_mode='r'); g = np.load(self.dir / k / 'glob.npy', mmap_mode='r'); n = arr.shape[0]
            assert arr.shape[1:] == (NN, NCH) and g.shape == (n, GCH), f'{k}: unexpected tensor shape {arr.shape} {g.shape}'
            for a in range(0, n, chunk):
                b = min(a + chunk, n); self.nodes[o + a:o + b] = torch.from_numpy(np.array(arr[a:b])).to(device)
            self.glob[o:o + n] = torch.from_numpy(np.array(g)).to(device)
            w = np.load(self.dir / k / 'win_end.npy'); wins.append(w.astype(np.int64) + o); sid.append(np.full(len(w), i, np.int16)); self.off[k] = (o, n); o += n; del arr, g
        self.win = np.concatenate(wins); self.win_set = np.concatenate(sid); self.ar = torch.arange(-SEQ + 1, 1, device=device)
    def gb(self): return (self.nodes.numel() * 2 + self.glob.numel() * 2) / 1e9
    def batch(self, g):
        """g: LongTensor of GLOBAL end indices t. Input = intervals t-49..t (inclusive). Target = interval t+1 only."""
        idx = g[:, None] + self.ar
        xn = self.nodes[idx].float(); xg = self.glob[idx].float()
        yn = self.nodes[g + 1][:, :, LOGQ].float(); yg = self.glob[g + 1][:, GLS].float()
        y = torch.cat([yn, yg[:, None]], 1); last = torch.cat([xn[:, -1, :, LOGQ], xg[:, -1, GLS:GLS + 1]], 1)
        return xn, xg, y, last

# ---------------------------------------------------------------- evaluation harness (per-window losses are kept for the later bootstrap)
@torch.no_grad()
def evaluate(bank, predict, bs=4096, extra=None):
    n = len(bank.win); loss = np.zeros(n, np.float32); chg = np.zeros(n, np.float32); nch = np.zeros(n, np.int16); spr = np.zeros(n, np.float32); ex = []
    for a in range(0, n, bs):
        b = min(a + bs, n); g = torch.from_numpy(bank.win[a:b]).to(bank.device); xn, xg, y, last = bank.batch(g)
        err = (predict(xn, xg, last) - y).abs(); m = (y != last)
        loss[a:b] = err.mean(1).cpu().numpy(); chg[a:b] = (err * m).sum(1).cpu().numpy(); nch[a:b] = m.sum(1).cpu().numpy(); spr[a:b] = err[:, NN].cpu().numpy()
        if extra is not None: ex.append(extra(xn, xg, last).cpu().numpy())
    out = {'loss': loss, 'chg_sum': chg, 'n_chg': nch, 'spr': spr}
    if extra is not None: out['extra'] = np.concatenate(ex)
    return out
def summarize(bank, res):
    rows = {}
    for i, k in enumerate(bank.keys):
        m = bank.win_set == i
        rows[k] = {'n_windows': int(m.sum()), 'mae_all': float(res['loss'][m].mean()), 'mae_changed': float(res['chg_sum'][m].sum() / max(int(res['n_chg'][m].sum()), 1)),
                   'share_changed': float(res['n_chg'][m].sum() / (NT * max(int(m.sum()), 1))), 'mae_spread': float(res['spr'][m].mean())}
    return rows
def average_results(rs):
    """seeds are averaged FIRST (mean of the per-seed losses of each window), exactly as in the frozen rule"""
    return {'loss': np.mean([r['loss'] for r in rs], 0).astype(np.float32), 'chg_sum': np.mean([r['chg_sum'] for r in rs], 0).astype(np.float32),
            'n_chg': rs[0]['n_chg'], 'spr': np.mean([r['spr'] for r in rs], 0).astype(np.float32)}
def save_result(res, bank, path):
    path = Path(path)
    if path.exists(): raise FileExistsError(path)
    tmp = path.with_name(path.stem + '.tmp.npz'); np.savez_compressed(tmp, win=bank.win, win_set=bank.win_set, **{k: v for k, v in res.items() if k != 'extra'}); os.replace(tmp, path)

# ---------------------------------------------------------------- baselines
def pred_b1(xn, xg, last): return last
def ema_weights(span, device='cpu'):
    a = 2.0 / (span + 1.0); j = torch.arange(SEQ - 1, -1, -1, dtype=torch.float32, device=device)       # time index 0 = oldest, 49 = latest
    w = (1 - a) ** j; return w / w.sum()
SPANS = [1, 2, 3, 5, 8, 13, 21, 34, 50]
def make_b2(span_node, span_spread):
    def f(xn, xg, last):
        wn = ema_weights(span_node, xn.device); ws = ema_weights(span_spread, xn.device)
        pn = torch.einsum('t,btn->bn', wn, xn[:, :, :, LOGQ]); ps = torch.einsum('t,bt->b', ws, xg[:, :, GLS])
        return torch.cat([pn, ps[:, None]], 1)
    return f
@torch.no_grad()
def tune_b2(bank, bs=4096):
    """one span for node targets, one span for the spread target, chosen on validation only"""
    sn = np.zeros(len(SPANS)); ss = np.zeros(len(SPANS)); n = len(bank.win)
    for a in range(0, n, bs):
        b = min(a + bs, n); g = torch.from_numpy(bank.win[a:b]).to(bank.device); xn, xg, y, last = bank.batch(g)
        for i, sp in enumerate(SPANS):
            p = make_b2(sp, sp)(xn, xg, last); e = (p - y).abs(); sn[i] += float(e[:, :NN].sum()); ss[i] += float(e[:, NN].sum())
    sn /= n * NN; ss /= n; return {'span_node': SPANS[int(sn.argmin())], 'span_spread': SPANS[int(ss.argmin())], 'val_mae_node_by_span': dict(zip(SPANS, sn.tolist())), 'val_mae_spread_by_span': dict(zip(SPANS, ss.tolist()))}
def pred_b4(xn, xg, last):
    """B4: the z-score centre as a naive predictor. Spread target -> window mean (z = 0). Node targets are not part of B4 (persistence is used only so that a tensor exists; B4 is judged on the spread column only)."""
    return torch.cat([last[:, :NN], xg[:, :, GLS].mean(1, keepdim=True)], 1)
def b4_score(xn, xg, last):
    """rolling z-score anomaly score of spread and top-5 imbalance (max of the two absolute z), saved for later fusion/anomaly use"""
    def z(v): return ((v[:, -1] - v.mean(1)) / (v.std(1) + 1e-3)).abs()
    return torch.maximum(z(xg[:, :, GLS]), z(xg[:, :, GIMB]))

# ---------------------------------------------------------------- B3 MLP (the same output rule is used for every learned model: prediction = value at t + learned change)
class MLP(nn.Module):
    def __init__(s, h):
        super().__init__(); s.net = nn.Sequential(nn.Linear(SEQ * NN * NCH + SEQ * GCH, h), nn.ReLU(), nn.Linear(h, h), nn.ReLU(), nn.Linear(h, NT))
        nn.init.zeros_(s.net[-1].weight); nn.init.zeros_(s.net[-1].bias)           # starts exactly at persistence
    def forward(s, xn, xg, last): return last + s.net(torch.cat([xn.flatten(1), xg.flatten(1)], 1))
def make_mlp(cfg): return MLP(cfg['hidden'])
GRID = [{'hidden': h, 'lr': lr} for h in (64, 128) for lr in (1e-3, 3e-4)]      # the 4 pre-listed configurations

# ---------------------------------------------------------------- resumable trainer (also used by D2 for the TGNN)
def _load_latest(out, model, opt):
    for p in sorted(out.glob('resume_e*.pt'), reverse=True):
        try: ck = torch.load(p, map_location='cpu', weights_only=False)
        except Exception: continue
        model.load_state_dict(ck['model']); opt.load_state_dict(ck['opt']); torch.set_rng_state(ck['rng_cpu'])
        if ck.get('rng_cuda') is not None and torch.cuda.is_available(): torch.cuda.set_rng_state_all(ck['rng_cuda'])
        return ck['state'], p.name
    return None, None
def fit(make_model, cfg, bank_tr, bank_va, out_dir, seed, frac=0.10, max_epochs=20, patience=3, bs=256, device='cpu', log=print, _halt_after=None):
    out = Path(out_dir); out.mkdir(parents=True, exist_ok=True)
    if (out / 'done.json').exists(): return json.loads((out / 'done.json').read_text())
    assert all(bank_tr.summ[k]['split'] == 'train' for k in bank_tr.keys), 'training bank contains a non-training set'
    torch.manual_seed(seed); model = make_model(cfg).to(device); opt = torch.optim.Adam(model.parameters(), lr=cfg['lr'])
    state = {'epoch': 0, 'best': float('inf'), 'best_epoch': 0, 'bad': 0, 'hist': []}
    st, name = _load_latest(out, model, opt)
    if st is not None: state = st; log(f'  resumed from {name} (epoch {state["epoch"]})')
    n = len(bank_tr.win)
    while state['epoch'] < max_epochs and state['bad'] < patience:
        e = state['epoch'] + 1; t0 = time.time(); rng = np.random.default_rng([seed, e]); pick = rng.permutation(n)[:int(n * frac)]
        model.train(); tot = torch.zeros((), device=device); cnt = 0
        for a in range(0, len(pick), bs):
            g = torch.from_numpy(bank_tr.win[pick[a:a + bs]]).to(device); xn, xg, y, last = bank_tr.batch(g)
            loss = (model(xn, xg, last) - y).abs().mean(); opt.zero_grad(set_to_none=True); loss.backward(); torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0); opt.step()
            tot += loss.detach() * len(g); cnt += len(g)
        t_train = time.time() - t0; model.eval(); res = evaluate(bank_va, lambda xn, xg, last: model(xn, xg, last)); val = float(res['loss'].mean())
        torch.save(model.state_dict(), out / f'weights_e{e:03d}.pt')
        state['epoch'] = e; state['hist'].append({'epoch': e, 'train_mae': float(tot / max(cnt, 1)), 'val_mae': val, 'train_seconds': round(t_train, 1), 'epoch_seconds': round(time.time() - t0, 1)})
        if val < state['best'] - 1e-9: state['best'] = val; state['best_epoch'] = e; state['bad'] = 0
        else: state['bad'] += 1
        tmp = out / f'resume_e{e:03d}.pt.tmp'
        torch.save({'model': model.state_dict(), 'opt': opt.state_dict(), 'rng_cpu': torch.get_rng_state(), 'rng_cuda': torch.cuda.get_rng_state_all() if torch.cuda.is_available() else None, 'state': state}, tmp); os.replace(tmp, out / f'resume_e{e:03d}.pt')
        for p in out.glob('resume_e*.pt'):                                  # keep the two newest optimiser states; weights of every epoch stay
            if int(p.stem[8:]) < e - 1: p.unlink()
        with open(out / 'log.jsonl', 'a') as f: f.write(json.dumps(state['hist'][-1]) + '\n')
        log(f'  epoch {e}: train {state["hist"][-1]["train_mae"]:.5f} | val {val:.5f} | best {state["best"]:.5f} (e{state["best_epoch"]}) | {state["hist"][-1]["epoch_seconds"]} s')
        if _halt_after is not None and e >= _halt_after: return None
    done = {'cfg': cfg, 'seed': seed, 'best_val': state['best'], 'best_epoch': state['best_epoch'], 'epochs_run': state['epoch'], 'history': state['hist'], 'n_params': int(sum(p.numel() for p in model.parameters())),
            'mean_epoch_seconds': float(np.mean([h['epoch_seconds'] for h in state['hist']])), 'frac': frac, 'max_epochs': max_epochs, 'patience': patience, 'batch_size': bs}
    dump_json(done, out / 'done.json'); return done
def load_best(make_model, cfg, out_dir, device='cpu'):
    d = json.loads((Path(out_dir) / 'done.json').read_text()); m = make_model(cfg).to(device); m.load_state_dict(torch.load(Path(out_dir) / f'weights_e{d["best_epoch"]:03d}.pt', map_location=device)); m.eval(); return m, d

# ---------------------------------------------------------------- synthetic Stage C folder (for the code tests only)
def make_mock_stagec(root, n_int=2600, seed=0, bad_gap=(1200, 1290)):
    root = Path(root); sets = root / 'sets'; (root / 'config').mkdir(parents=True, exist_ok=True); rng = np.random.default_rng(seed)
    dump_json({'scaler': {'mock': True}, 'sha256': canonical_sha({'mock': True})}, root / 'config' / 'scaler_train_only.json')
    for split in ('train', 'val', 'test'):
        for day in SPLITS[split]:
            for sym in SYMS:
                k = f'{sym}_{day}'; d = sets / k; d.mkdir(parents=True, exist_ok=True)
                lq = np.zeros((n_int, NN), np.float32); cur = rng.normal(size=NN)
                for i in range(n_int):
                    ch = rng.random(NN) < 0.25; cur = np.where(ch, 0.9 * cur + rng.normal(scale=0.5, size=NN), cur); lq[i] = cur
                nodes = rng.normal(size=(n_int, NN, NCH)).astype(np.float32); nodes[:, :, 0] = np.where(np.arange(NN) < 25, -1, 1); nodes[:, :, LOGQ] = lq
                gl = rng.normal(size=(n_int, GCH)).astype(np.float32); gl[1:, GLS] = np.where(rng.random(n_int - 1) < 0.2, rng.normal(size=n_int - 1), 0); gl[:, GLS] = np.maximum.accumulate(np.zeros(n_int)) + np.cumsum(np.where(rng.random(n_int) < .1, rng.normal(scale=.3, size=n_int), 0))
                flags = np.zeros(n_int, np.uint8); flags[:SEQ + 20] = 16; flags[bad_gap[0]:bad_gap[1]] = 8
                c = np.concatenate([[0], np.cumsum((flags != 0).astype(np.int64))]); t = np.arange(SEQ, n_int - 1); win = t[(c[t + 2] - c[t - SEQ]) == 0].astype(np.int32)
                np.save(d / 'nodes.npy', nodes.astype(np.float16)); np.save(d / 'glob.npy', gl.astype(np.float16)); np.save(d / 'flags.npy', flags); np.save(d / 'win_end.npy', win)
                dump_json({'sym': sym, 'day': day, 'split': split, 'n_intervals': n_int, 'n_windows': int(len(win)), 'scaler_sha256': canonical_sha({'mock': True}), 'hashes': {f: sha256_file(d / f) for f in ('nodes.npy', 'glob.npy', 'flags.npy', 'win_end.npy')}}, d / 'set_summary.json')
    return root

# ---------------------------------------------------------------- E-tests (code on synthetic data)
def unit_tests(tmp):
    tmp = Path(tmp); out = []
    def add(i, nm, ok, det=''): out.append((i, nm, bool(ok), str(det)))
    root = make_mock_stagec(tmp / 'sc', n_int=2600); fz = tmp / 'freeze.json'
    # E01 test days are refused before the freeze file exists
    try: Bank(root, keys_of('test')[:1], freeze_file=fz); refused = False
    except PermissionError: refused = True
    dump_json({'x': 1}, fz); ok_after = True
    try: Bank(root, keys_of('test')[:1], freeze_file=fz)
    except Exception: ok_after = False
    fz.unlink(); add('E01', 'test days refused until the freeze file exists', refused and ok_after, f'refused={refused} allowed_after={ok_after}')
    btr = Bank(root, keys_of('train'), device='cpu'); bva = Bank(root, keys_of('val'), device='cpu')
    # E02 exact window indexing against direct array access
    rng = np.random.default_rng(1); worst = 0.0; okidx = True
    for i in rng.choice(len(btr.win), 40, replace=False):
        gt = int(btr.win[i]); k = btr.keys[int(btr.win_set[i])]; o, n = btr.off[k]; t = gt - o
        raw = np.load(btr.dir / k / 'nodes.npy'); rg = np.load(btr.dir / k / 'glob.npy')
        xn, xg, y, last = btr.batch(torch.tensor([gt]))
        okidx &= np.array_equal(xn[0].numpy(), raw[t - 49:t + 1].astype(np.float32)) and np.array_equal(xg[0].numpy(), rg[t - 49:t + 1].astype(np.float32))
        okidx &= np.array_equal(y[0, :NN].numpy(), raw[t + 1, :, LOGQ].astype(np.float32)) and float(y[0, NN]) == float(rg[t + 1, GLS]) and np.array_equal(last[0, :NN].numpy(), raw[t, :, LOGQ].astype(np.float32))
    add('E02', 'input = intervals t-49..t, target = interval t+1 only (40 random windows, exact)', okidx)
    # E03 the future cannot leak: overwrite interval t+1 and t+2 and check the input does not change
    gt = int(btr.win[100]); a = btr.batch(torch.tensor([gt]))[0].clone(); y0 = btr.batch(torch.tensor([gt]))[2].clone()
    sv = btr.nodes[gt + 1:gt + 3].clone(); btr.nodes[gt + 1:gt + 3] = 77.0; b = btr.batch(torch.tensor([gt]))
    same_x = torch.equal(a, b[0]); y_changed = not torch.equal(y0, b[2]); btr.nodes[gt + 1:gt + 3] = sv; add('E03', 'changing t+1 and t+2 does not change the input, only the target', same_x and y_changed)
    # E04 no window touches a flagged interval, and no window crosses a day boundary
    okf = True; okb = True
    for i, k in enumerate(btr.keys):
        fl = np.load(btr.dir / k / 'flags.npy'); w = np.load(btr.dir / k / 'win_end.npy'); okf &= all((fl[t - SEQ:t + 2] == 0).all() for t in w[::7]); okb &= int(w.min()) >= SEQ and int(w.max()) + 1 < btr.off[k][1]
    add('E04', 'windows contain no flagged interval and stay inside their day', okf and okb)
    # E05 baselines
    bs_ = 512; res1 = evaluate(bva, pred_b1, bs=bs_); man = []
    for i in rng.choice(len(bva.win), 30, replace=False):
        xn, xg, y, last = bva.batch(torch.tensor([int(bva.win[i])])); man.append((i, float((last - y).abs().mean())))
    okb1 = all(abs(res1['loss'][i] - v) < 1e-6 for i, v in man)
    res2 = evaluate(bva, make_b2(1, 1)); w = ema_weights(8); okb2 = np.allclose(res2['loss'], res1['loss'], atol=1e-6) and abs(float(w.sum()) - 1) < 1e-6 and bool((w >= 0).all()) and bool(w[-1] == w.max())
    add('E05', 'B1 equals a hand computation; B2 with span 1 equals B1; EMA weights are valid and causal', okb1 and okb2)
    # E06 changed-node mask by hand
    xn = torch.zeros(1, SEQ, NN, NCH); xg = torch.zeros(1, SEQ, GCH); last = torch.zeros(1, NT); y = torch.zeros(1, NT); y[0, 3] = 1.0; y[0, 50] = 2.0; p = torch.zeros(1, NT); err = (p - y).abs(); m = (y != last)
    add('E06', 'changed-target mask and per-window sums', int(m.sum()) == 2 and float((err * m).sum()) == 3.0)
    # E07 MLP starts exactly at persistence and can learn
    torch.manual_seed(0); mlp = MLP(16); xn, xg, y, last = btr.batch(torch.tensor(btr.win[:8])); p0 = mlp(xn, xg, last); okz = torch.equal(p0, last)
    opt = torch.optim.Adam(mlp.parameters(), lr=1e-3); l0 = float((mlp(xn, xg, last) - y).abs().mean().detach())
    for _ in range(60): l = (mlp(xn, xg, last) - y).abs().mean(); opt.zero_grad(); l.backward(); opt.step()
    add('E07', 'B3 starts at persistence and its loss falls on a small fit', okz and float(l.detach()) < l0 * 0.9, f'{l0:.4f}->{float(l.detach()):.4f}')
    # E08 resume gives exactly the same result as an uninterrupted run
    cfg = {'hidden': 8, 'lr': 1e-3}; quiet = lambda *a, **k: None
    d1 = fit(make_mlp, cfg, btr, bva, tmp / 'r_full', seed=5, frac=0.2, max_epochs=3, patience=9, bs=64, log=quiet)
    fit(make_mlp, cfg, btr, bva, tmp / 'r_part', seed=5, frac=0.2, max_epochs=3, patience=9, bs=64, log=quiet, _halt_after=1)
    d2 = fit(make_mlp, cfg, btr, bva, tmp / 'r_part', seed=5, frac=0.2, max_epochs=3, patience=9, bs=64, log=quiet)
    sa = torch.load(tmp / 'r_full' / 'weights_e003.pt'); sb = torch.load(tmp / 'r_part' / 'weights_e003.pt')
    add('E08', 'a stopped-and-resumed run equals an uninterrupted run (weights and validation history)', all(torch.equal(sa[k], sb[k]) for k in sa) and [h['val_mae'] for h in d1['history']] == [h['val_mae'] for h in d2['history']])
    # E09 seed averaging and per-set summary by hand
    r = [{'loss': np.array([1., 2.], np.float32), 'chg_sum': np.array([2., 4.], np.float32), 'n_chg': np.array([1, 2], np.int16), 'spr': np.array([0., 1.], np.float32)}, {'loss': np.array([3., 4.], np.float32), 'chg_sum': np.array([4., 6.], np.float32), 'n_chg': np.array([1, 2], np.int16), 'spr': np.array([2., 3.], np.float32)}]
    av = average_results(r); add('E09', 'seeds are averaged per window before any statistic', np.allclose(av['loss'], [2, 3]) and np.allclose(av['chg_sum'], [3, 5]) and np.allclose(av['spr'], [1, 2]))
    # E10 sampler: deterministic per (seed, epoch), training bank holds only training sets, validation bank only validation
    p1 = np.random.default_rng([3, 2]).permutation(1000)[:100]; p2 = np.random.default_rng([3, 2]).permutation(1000)[:100]
    add('E10', 'epoch sample is deterministic per (seed, epoch); training bank = training sets only; validation bank = validation sets only', np.array_equal(p1, p2) and all(btr.summ[k]['split'] == 'train' for k in btr.keys) and all(bva.summ[k]['split'] == 'val' for k in bva.keys))
    # E11 frozen config
    try: check_frozen(tmp / 'fz.json', {'a': 1}); check_frozen(tmp / 'fz.json', {'a': 1}); refused = False; check_frozen(tmp / 'fz.json', {'a': 2})
    except RuntimeError: refused = True
    add('E11', 'a changed frozen config is refused', refused)
    # E12 put_json never overwrites
    put_json({'a': 1}, tmp / 'p.json'); put_json({'a': 2}, tmp / 'p.json'); add('E12', 'put_json keeps the first file and writes a changed one as .r1', json.loads((tmp / 'p.json').read_text()) == {'a': 1} and (tmp / 'p.r1.json').exists())
    # E13 B4 and tune_b2 run and give sane numbers
    tb = tune_b2(bva); sc_ = evaluate(bva, pred_b4, extra=b4_score); add('E13', 'B2 tuning and B4 run; B4 score is finite and non-negative', tb['span_node'] in SPANS and np.isfinite(sc_['extra']).all() and (sc_['extra'] >= 0).all())
    return out

Writing staged_lib.py


In [ ]:
%%writefile staged_d2_lib.py
"""D2 library: TGNN M1 (rank-chain + best-bid/best-ask + global edges), M0 (self + global links only), B3a (MLP with the same auxiliary loss),
resumable trainer with a time budget, change-occurrence AUC. Uses the frozen D1 library unchanged."""
import os, json, time
from pathlib import Path
import numpy as np
import torch, torch.nn as nn, torch.nn.functional as F
import staged_lib as S
from staged_lib import SEQ, NN, NCH, GCH, NT, LOGQ, GLS
NG = NN + 1                                   # 50 level nodes + 1 global node

# ---------------------------------------------------------------- graph
def build_adj():
    """row-normalised adjacency per edge type. Node order: bid rank 1..25 (0..24), ask rank 1..25 (25..49), global (50)."""
    chain = np.zeros((NG, NG), np.float32); spread = np.zeros((NG, NG), np.float32); glob = np.zeros((NG, NG), np.float32)
    for r in range(24):
        chain[r, r + 1] = chain[r + 1, r] = 1; chain[25 + r, 26 + r] = chain[26 + r, 25 + r] = 1
    spread[0, 25] = spread[25, 0] = 1
    glob[:NN, NN] = 1; glob[NN, :NN] = 1
    out = {}
    for k, a in (('chain', chain), ('spread', spread), ('glob', glob)):
        rs = a.sum(1, keepdims=True); out[k] = a / np.maximum(rs, 1)
    return out, {'chain': chain, 'spread': spread, 'glob': glob}
EDGES_M1 = ('chain', 'spread', 'glob'); EDGES_M0 = ('glob',)

class TGNN(nn.Module):
    def __init__(s, h=64, edges=EDGES_M1, layers=2):
        super().__init__(); s.h = h; s.edges = tuple(edges); s.layers = layers; adj, _ = build_adj()
        s.emb = nn.Embedding(NG, 8); s.enc_n = nn.Linear(NCH + 8, h); s.enc_g = nn.Linear(GCH + 8, h)
        s.self_lin = nn.ModuleList([nn.Linear(h, h) for _ in range(layers)]); s.norm = nn.ModuleList([nn.LayerNorm(h) for _ in range(layers)])
        s.edge_lin = nn.ModuleList([nn.ModuleDict({e: nn.Linear(h, h, bias=False) for e in s.edges}) for _ in range(layers)])
        for e in ('chain', 'spread', 'glob'): s.register_buffer('A_' + e, torch.from_numpy(adj[e]))
        s.gru = nn.GRU(h, h, batch_first=True)
        s.head_n = nn.Sequential(nn.Linear(2 * h, h), nn.ReLU(), nn.Linear(h, 2)); s.head_g = nn.Sequential(nn.Linear(2 * h, h), nn.ReLU(), nn.Linear(h, 1))
        for m in (s.head_n[-1], s.head_g[-1]): nn.init.zeros_(m.weight); nn.init.zeros_(m.bias)         # starts exactly at persistence
    def forward(s, xn, xg, last):
        B, T = xn.shape[0], xn.shape[1]; em = s.emb.weight
        n_in = torch.cat([xn, em[:NN][None, None].expand(B, T, -1, -1)], -1); g_in = torch.cat([xg[:, :, None, :], em[NN][None, None, None].expand(B, T, 1, -1)], -1)
        h = torch.cat([s.enc_n(n_in), s.enc_g(g_in)], 2)                                              # (B,T,51,h)
        for l in range(s.layers):
            agg = s.self_lin[l](h)
            for e in s.edges: agg = agg + s.edge_lin[l][e](torch.matmul(getattr(s, 'A_' + e).to(h.dtype), h))
            h = h + F.relu(s.norm[l](agg))
        seq = h.permute(0, 2, 1, 3).reshape(B * NG, T, s.h); out, _ = s.gru(seq)
        feat = torch.cat([out[:, -1], h[:, -1].reshape(B * NG, s.h)], 1).reshape(B, NG, 2 * s.h)
        o = s.head_n(feat[:, :NN]); dn = o[..., 0]; logit = o[..., 1]; dsp = s.head_g(feat[:, NN])[:, 0]
        return last + torch.cat([dn, dsp[:, None]], 1).float(), logit.float()
def make_m1(cfg): return TGNN(cfg['hidden'], EDGES_M1)
def make_m0(cfg): return TGNN(cfg['hidden'], EDGES_M0)

class MLPA(nn.Module):
    """B3a: the D1 MLP (same sizes) with an extra change-occurrence output, trained with the same loss as M0 and M1"""
    def __init__(s, h):
        super().__init__(); s.net = nn.Sequential(nn.Linear(SEQ * NN * NCH + SEQ * GCH, h), nn.ReLU(), nn.Linear(h, h), nn.ReLU(), nn.Linear(h, NT + NN))
        nn.init.zeros_(s.net[-1].weight); nn.init.zeros_(s.net[-1].bias)
    def forward(s, xn, xg, last):
        o = s.net(torch.cat([xn.flatten(1), xg.flatten(1)], 1)).float(); return last + o[:, :NT], o[:, NT:]
def make_mlpa(cfg): return MLPA(cfg['hidden'])

# ---------------------------------------------------------------- loss, validation helpers
def loss_fn(model_out, y, last, aux):
    pred, logit = model_out; l1 = (pred - y).abs().mean(); bce = F.binary_cross_entropy_with_logits(logit, (y[:, :NN] != last[:, :NN]).float()); return l1 + aux * bce, l1, bce
def amp_ctx(device, amp): return torch.autocast('cuda', dtype=torch.float16, enabled=bool(amp and device == 'cuda'))
@torch.no_grad()
def val_mae(bank, windows, model, device, amp, bs=512):
    model.eval(); tot = 0.0; n = 0
    for a in range(0, len(windows), bs):
        g = torch.from_numpy(windows[a:a + bs]).to(device); xn, xg, y, last = bank.batch(g)
        with amp_ctx(device, amp): pred, _ = model(xn, xg, last)
        tot += float((pred.float() - y).abs().mean(1).sum()); n += len(g)
    return tot / max(n, 1)
def val_subset(bank, frac=0.25, seed=12345):
    """fixed random share of the validation windows of every set (so both symbols are represented); sorted for reproducibility"""
    rng = np.random.default_rng(seed); parts = []
    for i in range(len(bank.keys)):
        w = bank.win[bank.win_set == i]; parts.append(np.sort(rng.choice(w, int(len(w) * frac), replace=False)))
    return np.concatenate(parts)
@torch.no_grad()
def change_auc(model, bank, windows, device, amp, bs=512, max_pairs=1_500_000):
    model.eval(); sc = []; lb = []; kept = 0
    for a in range(0, len(windows), bs):
        g = torch.from_numpy(windows[a:a + bs]).to(device); xn, xg, y, last = bank.batch(g)
        with amp_ctx(device, amp): _, logit = model(xn, xg, last)
        sc.append(logit.float().flatten().cpu().numpy()); lb.append((y[:, :NN] != last[:, :NN]).flatten().cpu().numpy()); kept += sc[-1].size
        if kept >= max_pairs: break
    s_ = np.concatenate(sc); l_ = np.concatenate(lb); p = 1 / (1 + np.exp(-np.clip(s_, -30, 30))); bce = float(-(l_ * np.log(p + 1e-9) + (~l_) * np.log(1 - p + 1e-9)).mean())
    from staged_diag_lib import auc
    return {'auc': auc(l_, s_), 'bce': bce, 'base_rate': float(l_.mean()), 'pairs': int(len(l_))}
def predict_fn(model, device, amp):
    def f(xn, xg, last):
        with amp_ctx(device, amp): return model(xn, xg, last)[0].float()
    return f

# ---------------------------------------------------------------- resumable trainer with a time budget
def fit2(make_model, cfg, bank_tr, bank_va, val_windows, out_dir, seed, aux=0.2, frac=0.10, max_epochs=20, patience=3, bs=128, device='cpu', amp=False, deadline=None, log=print, _halt_after=None):
    out = Path(out_dir); out.mkdir(parents=True, exist_ok=True)
    if (out / 'done.json').exists(): return json.loads((out / 'done.json').read_text())
    assert all(bank_tr.summ[k]['split'] == 'train' for k in bank_tr.keys), 'training bank contains a non-training set'
    torch.manual_seed(seed); model = make_model(cfg).to(device); opt = torch.optim.Adam(model.parameters(), lr=cfg['lr']); scaler = torch.amp.GradScaler('cuda', enabled=bool(amp and device == 'cuda'))
    state = {'epoch': 0, 'best': float('inf'), 'best_epoch': 0, 'bad': 0, 'hist': []}
    for p in sorted(out.glob('resume_e*.pt'), reverse=True):
        try: ck = torch.load(p, map_location='cpu', weights_only=False)
        except Exception: continue
        model.load_state_dict(ck['model']); opt.load_state_dict(ck['opt']); scaler.load_state_dict(ck['scaler']); torch.set_rng_state(ck['rng_cpu'])
        if ck.get('rng_cuda') is not None and torch.cuda.is_available(): torch.cuda.set_rng_state_all(ck['rng_cuda'])
        state = ck['state']; log(f'  resumed from {p.name} (epoch {state["epoch"]})'); break
    n = len(bank_tr.win)
    while state['epoch'] < max_epochs and state['bad'] < patience:
        if deadline is not None and time.time() > deadline: log('  time budget reached before this epoch: stop here, run again to continue'); return None
        e = state['epoch'] + 1; t0 = time.time(); rng = np.random.default_rng([seed, e]); pick = rng.permutation(n)[:int(n * frac)]
        model.train(); tot = torch.zeros((), device=device); cnt = 0
        for a in range(0, len(pick), bs):
            g = torch.from_numpy(bank_tr.win[pick[a:a + bs]]).to(device); xn, xg, y, last = bank_tr.batch(g)
            with amp_ctx(device, amp): out_ = model(xn, xg, last)
            loss, l1, bce = loss_fn(out_, y, last, aux); opt.zero_grad(set_to_none=True); scaler.scale(loss).backward(); scaler.unscale_(opt); torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0); scaler.step(opt); scaler.update()
            tot += l1.detach() * len(g); cnt += len(g)
        t_train = time.time() - t0; val = val_mae(bank_va, val_windows, model, device, amp); torch.save(model.state_dict(), out / f'weights_e{e:03d}.pt')
        state['epoch'] = e; state['hist'].append({'epoch': e, 'train_l1': float(tot / max(cnt, 1)), 'val_mae_sub': val, 'train_seconds': round(t_train, 1), 'epoch_seconds': round(time.time() - t0, 1)})
        if val < state['best'] - 1e-9: state['best'] = val; state['best_epoch'] = e; state['bad'] = 0
        else: state['bad'] += 1
        tmp = out / f'resume_e{e:03d}.pt.tmp'
        torch.save({'model': model.state_dict(), 'opt': opt.state_dict(), 'scaler': scaler.state_dict(), 'rng_cpu': torch.get_rng_state(), 'rng_cuda': torch.cuda.get_rng_state_all() if torch.cuda.is_available() else None, 'state': state}, tmp); os.replace(tmp, out / f'resume_e{e:03d}.pt')
        for p in out.glob('resume_e*.pt'):
            if int(p.stem[8:]) < e - 1: p.unlink()
        with open(out / 'log.jsonl', 'a') as f: f.write(json.dumps(state['hist'][-1]) + '\n')
        log(f'  epoch {e}: train L1 {state["hist"][-1]["train_l1"]:.5f} | val(sub) {val:.5f} | best {state["best"]:.5f} (e{state["best_epoch"]}) | {state["hist"][-1]["epoch_seconds"]} s')
        if _halt_after is not None and e >= _halt_after: return None
    done = {'cfg': cfg, 'seed': seed, 'best_val_sub': state['best'], 'best_epoch': state['best_epoch'], 'epochs_run': state['epoch'], 'history': state['hist'], 'n_params': int(sum(p.numel() for p in model.parameters())),
            'mean_epoch_seconds': float(np.mean([h['epoch_seconds'] for h in state['hist']])), 'frac': frac, 'max_epochs': max_epochs, 'patience': patience, 'batch_size': bs, 'aux_weight': aux, 'amp': bool(amp and device == 'cuda')}
    S.dump_json(done, out / 'done.json'); return done
def load_best2(make_model, cfg, out_dir, device='cpu'):
    d = json.loads((Path(out_dir) / 'done.json').read_text()); m = make_model(cfg).to(device); m.load_state_dict(torch.load(Path(out_dir) / f'weights_e{d["best_epoch"]:03d}.pt', map_location=device)); m.eval(); return m, d

# ---------------------------------------------------------------- tests
def unit_tests(tmp):
    import staged_diag_lib as D
    tmp = Path(tmp); out = []
    def add(i, nm, ok, det=''): out.append((i, nm, bool(ok), str(det)))
    adj, raw = build_adj(); n_chain = int(raw['chain'].sum() // 2); n_glob = int(raw['glob'].sum() // 2)
    add('G01', 'graph: 48 chain edges (24 per side), 1 best-bid/best-ask edge joining nodes 0 and 25, global linked to all 50 nodes, rows normalised', n_chain == 48 and raw['spread'][0, 25] == 1 and raw['spread'].sum() == 2 and n_glob == 50 and np.allclose(adj['glob'][:NN].sum(1), 1) and raw['chain'][24, 25] == 0, f'chain {n_chain}, global {n_glob}')
    root = D.make_signal_bank(tmp / 'sc', 'reverse', n_int=2600); btr = S.Bank(root, S.keys_of('train')); bva = S.Bank(root, S.keys_of('val'))
    xn, xg, y, last = btr.batch(torch.tensor(btr.win[:6])); torch.manual_seed(0); m1 = TGNN(16, EDGES_M1); m0 = TGNN(16, EDGES_M0)
    add('G02', 'M1 and M0 start exactly at persistence', torch.equal(m1(xn, xg, last)[0], last) and torch.equal(m0(xn, xg, last)[0], last))
    # G03: M1 with chain and spread weights set to zero equals M0 (the only difference is the graph structure)
    torch.manual_seed(1); a = TGNN(16, EDGES_M1); b = TGNN(16, EDGES_M0); sd = a.state_dict()
    for k in list(sd):
        if 'edge_lin' in k and ('chain' in k or 'spread' in k): sd[k] = torch.zeros_like(sd[k])
    a.load_state_dict(sd); bs_ = {k: v for k, v in sd.items() if not ('edge_lin' in k and ('chain' in k or 'spread' in k))}; b.load_state_dict(bs_, strict=True)
    for m in (a, b):
        for hd in (m.head_n[-1], m.head_g[-1]): nn.init.normal_(hd.weight, std=0.1)
    b.head_n[-1].load_state_dict(a.head_n[-1].state_dict()); b.head_g[-1].load_state_dict(a.head_g[-1].state_dict())
    add('G03', 'M1 with its chain and best-bid/best-ask weights set to zero gives the same output as M0', torch.allclose(a(xn, xg, last)[0], b(xn, xg, last)[0], atol=1e-5))
    # G04: with one spatial layer, a neighbour's input reaches a node in M1 but not in M0 (M0 only has global links)
    torch.manual_seed(2); p1 = TGNN(16, EDGES_M1, layers=1); p0 = TGNN(16, EDGES_M0, layers=1)
    for m in (p1, p0):
        for hd in (m.head_n[-1], m.head_g[-1]): nn.init.normal_(hd.weight, std=0.1)
    x2 = xn.clone(); x2[:, :, 6, :] += 1.0; d1 = (p1(xn, xg, last)[0][:, 5] - p1(x2, xg, last)[0][:, 5]).abs().max(); d0 = (p0(xn, xg, last)[0][:, 5] - p0(x2, xg, last)[0][:, 5]).abs().max()
    add('G04', 'one spatial layer: node 6 input changes node 5 output in M1 (chain edge) but not in M0', float(d1) > 1e-6 and float(d0) < 1e-7, f'M1 {float(d1):.2e} M0 {float(d0):.2e}')
    # G05: loss pieces by hand
    pred = torch.zeros(1, NT); yy = torch.zeros(1, NT); yy[0, 3] = 1; ll = torch.zeros(1, NT); lg = torch.zeros(1, NN); l_, l1, bce = loss_fn((pred, lg), yy, ll, 0.2)
    add('G05', 'loss = L1 + 0.2 x BCE with the change mask as label', abs(float(l1) - 1 / NT) < 1e-6 and abs(float(bce) - np.log(2)) < 1e-6 and abs(float(l_) - (1 / NT + 0.2 * np.log(2))) < 1e-6)
    # G06: stopped-and-resumed run equals uninterrupted run; time budget stop returns None and keeps files
    cfg = {'hidden': 8, 'lr': 2e-3}; quiet = lambda *a, **k: None; vs = val_subset(bva, 0.5)
    d1_ = fit2(make_m1, cfg, btr, bva, vs, tmp / 'full', seed=3, frac=0.15, max_epochs=3, patience=9, bs=64, log=quiet)
    fit2(make_m1, cfg, btr, bva, vs, tmp / 'part', seed=3, frac=0.15, max_epochs=3, patience=9, bs=64, log=quiet, _halt_after=1)
    r = fit2(make_m1, cfg, btr, bva, vs, tmp / 'dead', seed=3, frac=0.15, max_epochs=3, patience=9, bs=64, log=quiet, deadline=time.time() - 1)
    d2_ = fit2(make_m1, cfg, btr, bva, vs, tmp / 'part', seed=3, frac=0.15, max_epochs=3, patience=9, bs=64, log=quiet)
    sa = torch.load(tmp / 'full' / 'weights_e003.pt'); sb = torch.load(tmp / 'part' / 'weights_e003.pt')
    add('G06', 'resume equals an uninterrupted run; an exhausted time budget stops cleanly (returns None)', r is None and all(torch.equal(sa[k], sb[k]) for k in sa) and [h['val_mae_sub'] for h in d1_['history']] == [h['val_mae_sub'] for h in d2_['history']])
    # G07: the TGNN can learn a real signal (every change is undone): validation MAE falls clearly below persistence; M1 and B3a run through the same trainer
    cfg2 = {'hidden': 16, 'lr': 3e-3}; dm = fit2(make_m1, cfg2, btr, bva, vs, tmp / 'learn', seed=5, frac=0.4, max_epochs=4, patience=9, bs=64, log=quiet); base = float(S.evaluate(bva, S.pred_b1)['loss'].mean())
    da = fit2(make_mlpa, {'hidden': 16, 'lr': 3e-3}, btr, bva, vs, tmp / 'mlpa', seed=5, frac=0.4, max_epochs=2, patience=9, bs=64, log=quiet)
    add('G07', 'M1 beats persistence by more than 10% when the data hold a learnable signal; B3a runs with the same trainer', dm['best_val_sub'] < 0.9 * base and da['epochs_run'] == 2, f"M1 {dm['best_val_sub']:.4f} vs persistence {base:.4f}")
    # G08: validation subset: fixed, covers both symbols, only validation windows
    v1 = val_subset(bva, 0.25); v2 = val_subset(bva, 0.25); sets = {int(bva.win_set[np.searchsorted(bva.win, w)]) for w in v1[:50]} | {int(bva.win_set[np.searchsorted(bva.win, w)]) for w in v1[-50:]}
    add('G08', 'validation subset is deterministic, sorted, inside the validation bank and covers both symbols', np.array_equal(v1, v2) and np.isin(v1, bva.win).all() and len(sets) == 2, f'{len(v1)} of {len(bva.win)}')
    # G09: change AUC works on the learned model
    m, _ = load_best2(make_m1, cfg2, tmp / 'learn'); rn = D.make_signal_bank(tmp / 'nonebank', 'none', n_int=1500); bn = S.Bank(rn, S.keys_of('val')); ca = change_auc(m, bn, bn.win[:1500], 'cpu', False); add('G09', 'change-occurrence AUC runs and is finite on a trained model', np.isfinite(ca['auc']) and 0 <= ca['auc'] <= 1, ca)
    return out

Writing staged_d2_lib.py


In [ ]:
%%writefile stage_e_lib.py
"""TradeSense C03 - Stage E library: order-book manipulation INJECTOR (synthetic, labelled as synthetic).
Works on the frozen Stage C tensors (scaled float16). Levels are decoded to quantities, a pattern is added to the quantities of existing price levels
(prices, spread and mid never change), and ONLY the quantity-derived channels are recomputed with the same formulas as Stage C:
  node ch2 log_qty, ch3 d_log_qty, ch4 cum_log_depth, glob ch1 imbalance_top5   (ch1/ch5 distances and glob ch0/ch2 do not depend on quantities).
Intervals outside [start, end+1] of an event are copied untouched. Injection never targets a flagged interval. Training days are never injected."""
import json, os
from pathlib import Path
import numpy as np

K = 25; NN = 50; NCH = 6; GCH = 3; LOGQ, DLQ, CUM = 2, 3, 4; IMB = 1
SCALED_NODE = [1, 2, 3, 4, 5]
PATTERNS = {1: 'spoof', 2: 'layering', 3: 'quote_stuffing', 4: 'liquidity_vacuum'}
MAGS = {1: 'subtle', 2: 'medium', 3: 'easy'}
# magnitude parameters per pattern and level (1 subtle, 2 medium, 3 easy). multipliers are in units of the day-median quantity of that rank
PAR = {1: {1: 3.0, 2: 8.0, 3: 20.0}, 2: {1: 2.0, 2: 5.0, 3: 10.0}, 3: {1: 0.8, 2: 2.0, 3: 4.0}, 4: {1: 0.60, 2: 0.85, 3: 0.95}}
DUR = {1: (10, 100), 2: (20, 150), 3: (30, 100), 4: (10, 50)}          # in 100 ms intervals

# ---------------------------------------------------------------- decode / recompute
def _mu_sd(sc, ch): j = SCALED_NODE.index(ch); return float(sc['node_mean'][j]), float(sc['node_std'][j])
def decode_q(nodes, sc, a, b):
    """quantities of intervals [a,b) from the scaled log_qty channel (float64, >= 0)"""
    m, s = _mu_sd(sc, LOGQ); lq = nodes[a:b, :, LOGQ].astype(np.float64) * s + m; return np.maximum(np.expm1(lq), 0.0)
def encode_channels(q, q_prev, sc, glob_mean, glob_std):
    """scaled (log_qty, d_log_qty, cum_log_depth) of quantities q (m,50) given the quantities of the interval just before (50,) and the scaled imbalance (m,)"""
    lq = np.log1p(q); lq_prev = np.log1p(q_prev)[None]; dlq = lq - np.concatenate([lq_prev, lq[:-1]], 0)
    cum = np.concatenate([np.log1p(np.cumsum(q[:, :K], 1)), np.log1p(np.cumsum(q[:, K:], 1))], 1)
    s5b = q[:, :5].sum(1); s5a = q[:, K:K + 5].sum(1); imb = (s5b - s5a) / np.maximum(s5b + s5a, 1e-12)
    out = []
    for ch, arr in ((LOGQ, lq), (DLQ, dlq), (CUM, cum)): m, s = _mu_sd(sc, ch); out.append((arr - m) / s)
    return out[0], out[1], out[2], (imb - glob_mean) / glob_std

def day_ref(nodes, sc, flags, step=50):
    """per-rank median quantity of the day (sampled every `step` intervals, flag-free intervals only): the unit of injection magnitude"""
    idx = np.flatnonzero(flags[::step] == 0) * step
    m, s = _mu_sd(sc, LOGQ); lq = nodes[idx, :, LOGQ].astype(np.float64) * s + m; return np.median(np.maximum(np.expm1(lq), 0.0), 0)

# ---------------------------------------------------------------- pattern definitions: additive/multiplicative change of q over the event
def pattern_delta(q_span, pat, mag, side, rng, ref, dur):
    """returns the modified quantities q' for the intervals of the event (dur, 50)"""
    q = q_span.copy(); off = 0 if side == 0 else K; par = PAR[pat][mag]; meta = {}
    if pat == 1:                                                   # spoof: one large order a few ranks away from the touch, then cancelled
        r = int(rng.integers(1, 6)); q[:, off + r] += par * ref[off + r]; meta = {'ranks': [r]}
    elif pat == 2:                                                 # layering: 3-6 consecutive ranks, staggered placement, removed together
        L = int(rng.integers(3, 7)); r0 = int(rng.integers(1, 5)); ranks = list(range(r0, r0 + L)); meta = {'ranks': ranks}
        for i, r in enumerate(ranks):
            st = min(i * int(rng.integers(0, 5)), max(dur - 2, 0)); q[st:, off + r] += par * ref[off + r] * rng.uniform(0.7, 1.3)
    elif pat == 3:                                                 # quote stuffing: rapid on/off of quantity on the 10 best ranks
        on = (np.arange(dur) % 2 == 0); q[on, off:off + 10] *= (1.0 + par); meta = {'ranks': list(range(10))}
    elif pat == 4:                                                 # liquidity vacuum: the k best ranks lose most of their quantity
        k = int(rng.integers(3, 9)); q[:, off:off + k] *= (1.0 - par); meta = {'ranks': list(range(k))}
    return q, meta

# ---------------------------------------------------------------- placement
def place_events(flags, win_end, n_events, rng, margin=600, gap=600, durations=DUR, patterns=(1, 2, 3, 4), mags=(1, 2, 3)):
    """random event slots inside long flag-free stretches: [start, end] (inclusive), margin intervals of clean context before and 50 after, at least `gap` intervals between events.
    n_events = events per (pattern, magnitude) pair."""
    n = len(flags); bad = np.concatenate([[0], np.cumsum((flags != 0).astype(np.int64))]); taken = np.zeros(n, bool); ev = []
    jobs = [(p, m) for p in patterns for m in mags for _ in range(n_events)]; rng.shuffle(jobs)
    for p, m in jobs:
        dur = int(rng.integers(durations[p][0], durations[p][1] + 1))
        for _try in range(200):
            s = int(rng.integers(margin, n - dur - 60)); e = s + dur - 1
            if bad[e + 52] - bad[s - margin] != 0: continue                # whole stretch must be flag-free
            if taken[max(s - gap, 0):e + gap].any(): continue
            taken[s:e + 1] = True; ev.append({'start': s, 'end': e, 'pattern': p, 'mag': m, 'side': int(rng.integers(0, 2))}); break
    ev.sort(key=lambda d: d['start']); return ev

# ---------------------------------------------------------------- apply
def inject_day(nodes, glob, flags, sc, events, seed):
    """returns copies (nodes2, glob2) and an interval label array; events must come from place_events (flag-free, separated)"""
    nodes2 = nodes.copy(); glob2 = glob.copy(); n = len(flags); pat_arr = np.zeros(n, np.int8); mag_arr = np.zeros(n, np.int8); rng = np.random.default_rng(seed)
    ref = day_ref(nodes, sc, flags); gm = float(sc['glob_mean'][IMB]); gs = float(sc['glob_std'][IMB]); out = []
    for e in events:
        s, t, p, m, side = e['start'], e['end'], e['pattern'], e['mag'], e['side']; dur = t - s + 1
        q = decode_q(nodes, sc, s - 1, t + 2)                          # rows: s-1 (context), s..t (event), t+1 (cancel interval)
        q_ev, meta = pattern_delta(q[1:1 + dur], p, m, side, rng, ref, dur)
        qn = q.copy(); qn[1:1 + dur] = q_ev                              # interval t+1 keeps the original (cancelled) quantities
        lq, dl, cu, im = encode_channels(qn[1:], q[0], sc, gm, gs)
        nodes2[s:t + 2, :, LOGQ] = lq; nodes2[s:t + 2, :, DLQ] = dl; nodes2[s:t + 2, :, CUM] = cu; glob2[s:t + 2, IMB] = im
        pat_arr[s:t + 1] = p; mag_arr[s:t + 1] = m; out.append({**e, **meta, 'pattern_name': PATTERNS[p], 'mag_name': MAGS[m]})
    return nodes2, glob2, pat_arr, mag_arr, out

def build_injected_set(src_dir, dst_dir, sc, n_per_pair, seed, margin=600, gap=600):
    """one Stage C set -> injected set folder with the same layout plus event_pattern.npy, event_mag.npy, events.json. Training sets are refused."""
    src = Path(src_dir); dst = Path(dst_dir); sm = json.loads((src / 'set_summary.json').read_text())
    if sm['split'] == 'train': raise PermissionError('training sets are never injected')
    nodes = np.load(src / 'nodes.npy', mmap_mode='r'); glob = np.load(src / 'glob.npy'); flags = np.load(src / 'flags.npy'); win = np.load(src / 'win_end.npy'); rng = np.random.default_rng(seed)
    ev = place_events(flags, win, n_per_pair, rng, margin=margin, gap=gap); n2, g2, pa, ma, evl = inject_day(np.asarray(nodes), glob, flags, sc, ev, seed + 1)
    dst.mkdir(parents=True, exist_ok=True); np.save(dst / 'nodes.npy', n2.astype(np.float16)); np.save(dst / 'glob.npy', g2.astype(np.float16)); np.save(dst / 'flags.npy', flags); np.save(dst / 'win_end.npy', win)
    np.save(dst / 'event_pattern.npy', pa); np.save(dst / 'event_mag.npy', ma); (dst / 'events.json').write_text(json.dumps(evl, indent=1))
    sm2 = dict(sm); sm2.update({'injected': True, 'synthetic_events': len(evl), 'source_set': str(src.name), 'injection_seed': seed}); (dst / 'set_summary.json').write_text(json.dumps(sm2, indent=1)); return evl

# ---------------------------------------------------------------- tests (synthetic data, no real data involved)
def make_toy(n=6000, seed=0):
    """a tiny consistent Stage-C-like day built with the SAME formulas: random quantities -> scaled channels (so decode/encode must be exact up to float16)"""
    rng = np.random.default_rng(seed); q = np.exp(rng.normal(3.0, 0.8, size=(n, NN))); q = np.where(rng.random((n, NN)) < 0.08, q * rng.uniform(1.5, 4, size=(n, NN)), q)
    for i in range(1, n):
        keep = rng.random(NN) < 0.8; q[i] = np.where(keep, q[i - 1], q[i])                                    # sparse changes
    sc = {'node_mean': [0.0, 3.0, 0.0, 6.0, 0.0], 'node_std': [1.0, 1.3, 0.8, 1.5, 1.0], 'glob_mean': [0.0, 0.0, 0.0], 'glob_std': [1.0, 0.4, 1.0]}
    nodes = np.zeros((n, NN, NCH), np.float32); glob = np.zeros((n, GCH), np.float32); lq, dl, cu, im = encode_channels(q, q[0], sc, 0.0, 0.4)
    nodes[:, :, 0] = np.where(np.arange(NN) < K, -1, 1); nodes[:, :, LOGQ] = lq; nodes[:, :, DLQ] = dl; nodes[:, :, CUM] = cu; nodes[:, :, 1] = rng.normal(size=(n, NN)); nodes[:, :, 5] = rng.normal(size=(n, NN)); glob[:, IMB] = im
    flags = np.zeros(n, np.uint8); flags[:700] = 16; flags[6000:6040] = 8
    return nodes.astype(np.float16), glob.astype(np.float16), flags, sc, q
def unit_tests(tmp):
    tmp = Path(tmp); out = []
    def add(i, nm, ok, det=''): out.append((i, nm, bool(ok), str(det)))
    nodes, glob, flags, sc, q_true = make_toy(n=12000); n = len(flags); rng = np.random.default_rng(1)
    # H01 decode gives back the quantities (float16 tolerance)
    qd = decode_q(nodes, sc, 0, n); rel = np.abs(qd - q_true) / np.maximum(q_true, 1.0); add('H01', 'decoded quantities equal the true quantities (float16 tolerance)', float(np.median(rel)) < 2e-3 and float(rel.max()) < 3e-2, f'median {np.median(rel):.1e} max {rel.max():.1e}')
    # H02 re-encoding unchanged quantities reproduces the stored channels
    lq, dl, cu, im = encode_channels(qd[1000:1100], qd[999], sc, 0.0, 0.4); add('H02', 'recomputed log_qty, d_log_qty, cum_log_depth and imbalance of untouched data equal the stored channels', np.abs(lq - nodes[1000:1100, :, LOGQ].astype(np.float64)).max() < 0.02 and np.abs(dl - nodes[1000:1100, :, DLQ].astype(np.float64)).max() < 0.05 and np.abs(cu - nodes[1000:1100, :, CUM].astype(np.float64)).max() < 0.05 and np.abs(im - glob[1000:1100, IMB].astype(np.float64)).max() < 0.05)
    # H03 placement rules
    ev = place_events(flags, None, 3, rng, margin=100, gap=100); ok = len(ev) == 36 and all(e['start'] >= 100 for e in ev); bad = np.concatenate([[0], np.cumsum(flags != 0)])
    ok2 = all(bad[e['end'] + 52] - bad[e['start'] - 100] == 0 for e in ev); ok3 = all(ev[i + 1]['start'] - ev[i]['end'] >= 100 for i in range(len(ev) - 1)); add('H03', '36 events placed (3 per pattern/magnitude), never over flagged intervals, with clean context and gaps', ok and ok2 and ok3, f'{len(ev)} events')
    # H04 untouched outside the event: intervals before start and after end+1 are identical, all pre-existing channels except the four recomputed ones unchanged
    n2, g2, pa, ma, evl = inject_day(nodes, glob, flags, sc, ev, 5); idx = np.arange(n); inside = np.zeros(n, bool)
    for e in ev: inside[e['start']:e['end'] + 2] = True
    same = np.array_equal(n2[~inside], nodes[~inside]) and np.array_equal(g2[~inside], glob[~inside]); other = np.array_equal(n2[:, :, [0, 1, 5]], nodes[:, :, [0, 1, 5]]) and np.array_equal(g2[:, [0, 2]], glob[:, [0, 2]])
    add('H04', 'intervals outside every event are byte-identical; distance, side and spread/mid channels are never changed', same and other)
    # H05 causality: the first changed interval is the event start; nothing before it changes
    first = [int(np.flatnonzero((n2[e['start'] - 100:e['end'] + 2] != nodes[e['start'] - 100:e['end'] + 2]).any((1, 2)))[0]) - 100 for e in ev[:12]]; add('H05', 'the first changed interval of every event is its start (nothing earlier is touched)', all(f == 0 for f in first), first[:6])
    # H06 spoof by hand: extra quantity on the chosen rank equals magnitude x day reference, restored after the event
    ref = day_ref(nodes, sc, flags); sp = [e for e in evl if e['pattern'] == 1][0]; s0, e0 = sp['start'], sp['end']; r = sp['ranks'][0] + (0 if sp['side'] == 0 else K)
    qa = decode_q(n2, sc, s0 - 1, e0 + 3); qb = decode_q(nodes, sc, s0 - 1, e0 + 3); add_q = (qa[1:1 + (e0 - s0 + 1), r] - qb[1:1 + (e0 - s0 + 1), r]); target = PAR[1][sp['mag']] * ref[r]
    add('H06', 'spoof: the added quantity equals the magnitude x day reference of that rank during the whole event and is gone after it', abs(np.median(add_q) - target) / target < 0.03 and abs(qa[-1, r] - qb[-1, r]) / max(qb[-1, r], 1) < 0.03, f'median {np.median(add_q):.1f} target {target:.1f}')
    # H07 vacuum removes quantity, stuffing alternates, layering adds on 3+ ranks
    vac = [e for e in evl if e['pattern'] == 4][0]; qv = decode_q(n2, sc, vac['start'], vac['end'] + 1); qo = decode_q(nodes, sc, vac['start'], vac['end'] + 1); off = 0 if vac['side'] == 0 else K; k = len(vac['ranks'])
    ratio = qv[:, off:off + k].sum() / max(qo[:, off:off + k].sum(), 1e-9); stf = [e for e in evl if e['pattern'] == 3][0]; qs = decode_q(n2, sc, stf['start'], stf['start'] + 6); qso = decode_q(nodes, sc, stf['start'], stf['start'] + 6); o2 = 0 if stf['side'] == 0 else K
    alt = (qs[:, o2] / np.maximum(qso[:, o2], 1e-9)); lay = [e for e in evl if e['pattern'] == 2][0]
    add('H07', 'vacuum keeps the expected share of quantity, stuffing alternates on/off, layering touches 3 to 6 ranks', abs(ratio - (1 - PAR[4][vac['mag']])) < 0.06 and alt[0] > 1.5 and abs(alt[1] - 1) < 0.05 and 3 <= len(lay['ranks']) <= 6, f'vacuum kept {ratio:.2f}; stuffing {alt[:3].round(2).tolist()}')
    # H08 labels, determinism, training refusal
    lab_ok = all((pa[e['start']:e['end'] + 1] == e['pattern']).all() and (ma[e['start']:e['end'] + 1] == e['mag']).all() for e in evl) and int((pa > 0).sum()) == sum(e['end'] - e['start'] + 1 for e in evl)
    n3, g3, *_ = inject_day(nodes, glob, flags, sc, ev, 5); det = np.array_equal(n2, n3) and np.array_equal(g2, g3)
    d = tmp / 'src_train'; d.mkdir(parents=True, exist_ok=True); (d / 'set_summary.json').write_text(json.dumps({'split': 'train'})); refused = False
    try: build_injected_set(d, tmp / 'dst', sc, 1, 0)
    except PermissionError: refused = True
    add('H08', 'interval labels equal the events, the same seed gives identical tensors, training sets are refused', lab_ok and det and refused)
    # H09 end-to-end on a folder: val-like set written, readable by the D1 loader layout, injected count as expected
    sd = tmp / 'src_val'; sd.mkdir(parents=True, exist_ok=True); np.save(sd / 'nodes.npy', nodes); np.save(sd / 'glob.npy', glob); np.save(sd / 'flags.npy', flags); np.save(sd / 'win_end.npy', np.arange(700, n - 2, 5).astype(np.int32)); (sd / 'set_summary.json').write_text(json.dumps({'split': 'val', 'n_intervals': n}))
    evx = build_injected_set(sd, tmp / 'dst_val', sc, 2, 3, margin=100, gap=100); fl = sorted(p.name for p in (tmp / 'dst_val').iterdir()); add('H09', 'a validation set is written with labels and events; summary says injected', len(evx) == 24 and 'event_pattern.npy' in fl and json.loads((tmp / 'dst_val' / 'set_summary.json').read_text())['injected'] is True, fl)
    return out

Writing stage_e_lib.py


In [ ]:
%%writefile stage_f_lib.py
"""TradeSense C03 - Stage F library: injected sets for TRAINING a detector (train days) and for testing it on shifted data (validation days).
Builds on the frozen Stage E injector (stage_e_lib, unchanged). Additions: a per-event magnitude scale, a duration multiplier, and one extra pattern
(5 = flicker: an order that is placed and pulled repeatedly) that is used ONLY in the shifted validation sets, never in training.
Test days are refused. Everything produced here is synthetic and labelled as such."""
import json
from pathlib import Path
import numpy as np
import stage_e_lib as E

NAMES = {**E.PATTERNS, 5: 'flicker'}
FLICKER_DUR = (60, 300)

def scaled_par(p, m, scale):
    b = E.PAR[p][m]
    return float(np.clip(b * scale, 0.30, 0.98)) if p == 4 else float(b * scale)

def delta_flicker(q_span, mag, side, rng, ref, dur, scale):
    q = q_span.copy(); off = 0 if side == 0 else E.K; par = E.PAR[1][mag] * scale; r = int(rng.integers(1, 6)); on = np.zeros(dur, bool); i = 0; state = True
    while i < dur:
        L = int(rng.integers(5, 16)) if state else int(rng.integers(3, 9)); on[i:i + L] = state; i += L; state = not state
    q[on, off + r] += par * ref[off + r]; return q, {'ranks': [r]}

def place_events_f(flags, n_per_pair, rng, patterns, scale_spec, dur_mult=1.0, margin=600, gap=600):
    """E.place_events with duration multiplier and a magnitude scale per event. scale_spec: ('loguniform', lo, hi) or ('choice', [..])"""
    dur = {p: (max(int(a * dur_mult), 3), max(int(b * dur_mult), 4)) for p, (a, b) in {**E.DUR, 5: FLICKER_DUR}.items()}
    ev = E.place_events(flags, None, n_per_pair, rng, margin=margin, gap=gap, durations=dur, patterns=tuple(patterns))
    for e in ev:
        e['scale'] = float(np.exp(rng.uniform(np.log(scale_spec[1]), np.log(scale_spec[2])))) if scale_spec[0] == 'loguniform' else float(rng.choice(scale_spec[1]))
    return ev

def inject_day_f(nodes, glob, flags, sc, events, seed):
    nodes2 = nodes.copy(); glob2 = glob.copy(); n = len(flags); pat_arr = np.zeros(n, np.int8); mag_arr = np.zeros(n, np.int8); rng = np.random.default_rng(seed)
    ref = E.day_ref(nodes, sc, flags); gm = float(sc['glob_mean'][E.IMB]); gs = float(sc['glob_std'][E.IMB]); out = []; saved = {p: dict(E.PAR[p]) for p in E.PAR}
    try:
        for e in events:
            s, t, p, m, side, scale = e['start'], e['end'], e['pattern'], e['mag'], e['side'], e['scale']; dur = t - s + 1
            q = E.decode_q(nodes, sc, s - 1, t + 2)
            if p == 5: q_ev, meta = delta_flicker(q[1:1 + dur], m, side, rng, ref, dur, scale)
            else:
                E.PAR[p][m] = scaled_par(p, m, scale); q_ev, meta = E.pattern_delta(q[1:1 + dur], p, m, side, rng, ref, dur); E.PAR[p][m] = saved[p][m]
            qn = q.copy(); qn[1:1 + dur] = q_ev
            lq, dl, cu, im = E.encode_channels(qn[1:], q[0], sc, gm, gs)
            nodes2[s:t + 2, :, E.LOGQ] = lq; nodes2[s:t + 2, :, E.DLQ] = dl; nodes2[s:t + 2, :, E.CUM] = cu; glob2[s:t + 2, E.IMB] = im
            pat_arr[s:t + 1] = p; mag_arr[s:t + 1] = m; out.append({**e, **meta, 'pattern_name': NAMES[p], 'mag_name': E.MAGS[m]})
    finally:
        for p in saved: E.PAR[p].update(saved[p])
    return nodes2, glob2, pat_arr, mag_arr, out

def build_set_f(src_dir, dst_dir, sc, n_per_pair, seed, patterns, scale_spec, dur_mult=1.0, margin=600, gap=600, allow_train=False):
    src = Path(src_dir); dst = Path(dst_dir); sm = json.loads((src / 'set_summary.json').read_text())
    if sm['split'] == 'test': raise PermissionError('test days are never injected here')
    if sm['split'] == 'train' and not allow_train: raise PermissionError('training sets need allow_train=True')
    if sm['split'] == 'val' and allow_train: raise PermissionError('allow_train is only for training sets')
    if sm['split'] == 'train' and 5 in patterns: raise PermissionError('pattern 5 (flicker) is held out and never used in training')
    nodes = np.load(src / 'nodes.npy', mmap_mode='r'); glob = np.load(src / 'glob.npy'); flags = np.load(src / 'flags.npy'); win = np.load(src / 'win_end.npy'); rng = np.random.default_rng(seed)
    ev = place_events_f(flags, n_per_pair, rng, patterns, scale_spec, dur_mult, margin, gap); n2, g2, pa, ma, evl = inject_day_f(np.asarray(nodes), glob, flags, sc, ev, seed + 1)
    dst.mkdir(parents=True, exist_ok=True); np.save(dst / 'nodes.npy', n2.astype(np.float16)); np.save(dst / 'glob.npy', g2.astype(np.float16)); np.save(dst / 'flags.npy', flags); np.save(dst / 'win_end.npy', win)
    np.save(dst / 'event_pattern.npy', pa); np.save(dst / 'event_mag.npy', ma); (dst / 'events.json').write_text(json.dumps(evl, indent=1))
    sm2 = dict(sm); sm2.update({'injected': True, 'synthetic_events': len(evl), 'source_set': src.name, 'injection_seed': seed, 'patterns': list(patterns), 'dur_mult': dur_mult}); (dst / 'set_summary.json').write_text(json.dumps(sm2, indent=1)); return evl

# ---------------------------------------------------------------- tests
def unit_tests(tmp):
    tmp = Path(tmp); out = []
    def add(i, nm, ok, det=''): out.append((i, nm, bool(ok), str(det)))
    nodes, glob, flags, sc, q = E.make_toy(n=30000); n = len(flags); rng = np.random.default_rng(2)
    ev = place_events_f(flags, 1, rng, (1, 2, 3, 4), ('choice', [1.0]), margin=100, gap=100)
    add('L01', 'placement gives 12 events with a scale each; durations follow the multiplier', len(ev) == 12 and all('scale' in e for e in ev) and all(E.DUR[e['pattern']][0] <= e['end'] - e['start'] + 1 <= E.DUR[e['pattern']][1] for e in ev))
    ev2 = place_events_f(flags, 1, np.random.default_rng(2), (1, 2, 3, 4), ('choice', [1.0]), dur_mult=2.0, margin=100, gap=100); add('L02', 'duration multiplier 2 makes events longer on average', np.mean([e['end'] - e['start'] for e in ev2]) > 1.4 * np.mean([e['end'] - e['start'] for e in ev]))
    spo = [dict(e) for e in ev if e['pattern'] == 1][:1]; res = []
    for sca in (1.0, 2.0):
        for e in spo: e['scale'] = sca
        n2, g2, pa, ma, evl = inject_day_f(nodes, glob, flags, sc, spo, 9); e0 = evl[0]; r_ = e0['ranks'][0] + (0 if e0['side'] == 0 else E.K)
        qa = E.decode_q(n2, sc, e0['start'], e0['end'] + 1)[:, r_]; qb = E.decode_q(nodes, sc, e0['start'], e0['end'] + 1)[:, r_]; res.append(float(np.median(qa - qb)))
    add('L03', 'spoof with scale 2 adds about twice the extra quantity of scale 1', 1.7 < res[1] / max(res[0], 1e-9) < 2.3, f'ratio {res[1] / max(res[0], 1e-9):.2f}')
    fl = [{'start': 2000, 'end': 2199, 'pattern': 5, 'mag': 3, 'side': 0, 'scale': 1.0}]; n3, g3, pa3, ma3, ev3 = inject_day_f(nodes, glob, flags, sc, fl, 4); r5 = ev3[0]['ranks'][0]
    d = E.decode_q(n3, sc, 2000, 2200)[:, r5] - E.decode_q(nodes, sc, 2000, 2200)[:, r5]; on = d > 0.5 * d.max(); sw = int((np.diff(on.astype(int)) != 0).sum())
    add('L04', 'flicker switches on and off several times and the extra quantity is zero when off', sw >= 6 and float(np.abs(d[~on]).max()) < 0.2 * d.max(), f'{sw} switches')
    same = np.array_equal(np.delete(n3, np.arange(2000, 2202), 0), np.delete(nodes, np.arange(2000, 2202), 0)); add('L05', 'nothing outside the event changes', same)
    import json as _j
    def mkset(split):
        d = Path(tmp) / f'src_{split}'; d.mkdir(parents=True, exist_ok=True); np.save(d / 'nodes.npy', nodes); np.save(d / 'glob.npy', glob); np.save(d / 'flags.npy', flags); np.save(d / 'win_end.npy', np.arange(700, n - 10, 3).astype(np.int32))
        (d / 'set_summary.json').write_text(_j.dumps({'split': split, 'n_intervals': n})); return d
    ok = {}
    for split, kw in (('test', {}), ('train', {}), ('val', {'allow_train': True})):
        try: build_set_f(mkset(split), Path(tmp) / f'dst_{split}', sc, 1, 1, (1, 2), ('choice', [1.0]), margin=100, gap=100, **kw); ok[split + str(kw)] = False
        except PermissionError: ok[split + str(kw)] = True
    try: build_set_f(mkset('train'), Path(tmp) / 'dst_t5', sc, 1, 1, (1, 5), ('choice', [1.0]), margin=100, gap=100, allow_train=True); ok['flick_train'] = False
    except PermissionError: ok['flick_train'] = True
    add('L06', 'test days refused; training needs allow_train; validation cannot use allow_train; flicker refused in training', all(ok.values()), ok)
    evt = build_set_f(mkset('train'), Path(tmp) / 'dst_train_ok', sc, 1, 5, (1, 2, 3, 4), ('loguniform', 0.5, 2.0), margin=100, gap=100, allow_train=True); pa = np.load(Path(tmp) / 'dst_train_ok' / 'event_pattern.npy')
    add('L07', 'label array marks exactly the event intervals and the scales lie in range', int((pa > 0).sum()) == sum(e['end'] - e['start'] + 1 for e in evt) and all(0.5 <= e['scale'] <= 2.0 for e in evt) and len({round(e['scale'], 3) for e in evt}) > 6)
    a = build_set_f(mkset('val'), Path(tmp) / 'v1', sc, 1, 7, (1, 2, 3, 4, 5), ('choice', [0.6, 1.5]), margin=100, gap=100); b = build_set_f(mkset('val'), Path(tmp) / 'v2', sc, 1, 7, (1, 2, 3, 4, 5), ('choice', [0.6, 1.5]), margin=100, gap=100)
    add('L08', 'same seed gives identical sets; shifted validation set contains all five patterns', np.array_equal(np.load(Path(tmp) / 'v1' / 'nodes.npy'), np.load(Path(tmp) / 'v2' / 'nodes.npy')) and {e['pattern'] for e in a} == {1, 2, 3, 4, 5})
    return out

Writing stage_f_lib.py


In [ ]:
%%writefile stage_e2_lib.py
"""TradeSense C03 - Stage E2 library: scoring and evaluation of order-book irregularity scores on the injected validation sets.
A score s[i] belongs to TARGET interval i (it may only use intervals <= i). Alarm burden is matched on CLEAN intervals only (no event within 30 s)."""
import json
from pathlib import Path
import numpy as np, pandas as pd

NN = 50; LOGQ = 2; GAP = 100          # alarm intervals closer than 10 s form one episode
NEAR = 300                            # clean = at least 30 s away from any event

# ---------------------------------------------------------------- scores (all causal: s[i] uses data up to interval i)
def node_abs_change(nodes, a=None, b=None):
    """|change of the scaled log_qty channel| per node from interval i-1 to i, rows a..b-1 (row 0 of the day = 0); float32 (m,50)"""
    a = 0 if a is None else a; b = len(nodes) if b is None else b; lo = max(a - 1, 0); x = np.asarray(nodes[lo:b, :, LOGQ], np.float32); d = np.zeros_like(x); d[1:] = np.abs(x[1:] - x[:-1])
    return d[(a - lo):]
def train_node_mae(sets_dirs, flags_list, chunk=200_000):
    """per-node mean |change| over eligible targets of the TRAINING days (the standardisation of the persistence error)"""
    tot = np.zeros(NN); cnt = 0
    for d, fl in zip(sets_dirs, flags_list):
        nodes = np.load(Path(d) / 'nodes.npy', mmap_mode='r'); win = np.load(Path(d) / 'win_end.npy'); tgt = np.zeros(len(fl), bool); tgt[win + 1] = True
        for a in range(0, len(fl), chunk):
            b = min(a + chunk, len(fl)); dd = node_abs_change(nodes, a, b); m = tgt[a:b]; tot += dd[m].sum(0); cnt += int(m.sum())
    return tot / max(cnt, 1)
def score_persistence(nodes, mae, top=5):
    """standardised persistence error per node, then (a) mean of the `top` largest, (b) mean of all nodes; returns two float32 arrays (n,)"""
    n = len(nodes); s_top = np.zeros(n, np.float32); s_all = np.zeros(n, np.float32); CH = 200_000
    for a in range(0, n, CH):
        b = min(a + CH, n); e = node_abs_change(nodes, a, b) / np.maximum(mae, 1e-9)[None]; s_all[a:b] = e.mean(1); s_top[a:b] = np.partition(e, NN - top, axis=1)[:, NN - top:].mean(1)
    return s_top, s_all
def score_activity_z(nodes, win=600):
    """B5: causal robust z-score of the total absolute quantity change A_i = sum_j |change_j(i)| against the previous `win` intervals (median and MAD, current interval excluded)"""
    A = np.zeros(len(nodes), np.float64); CH = 200_000
    for a in range(0, len(nodes), CH): b = min(a + CH, len(nodes)); A[a:b] = node_abs_change(nodes, a, b).sum(1)
    ser = pd.Series(A); med = ser.rolling(win, min_periods=win // 2).median().shift(1); dev = (ser - med).abs(); mad = dev.rolling(win, min_periods=win // 2).median().shift(1)
    z = (ser - med) / (1.4826 * mad + 1e-6); return np.nan_to_num(z.to_numpy(), nan=-1e9).astype(np.float32)

# ---------------------------------------------------------------- masks, episodes, threshold, detection
def eligible_targets(win_end, n): m = np.zeros(n, bool); m[np.asarray(win_end) + 1] = True; return m
def clean_mask(n, events, near=NEAR):
    m = np.ones(n, bool)
    for e in events: m[max(e['start'] - near, 0):min(e['end'] + 2 + near, n)] = False
    return m
def episodes(alarm_idx, gap=GAP):
    """number of episodes: alarm intervals (sorted indices) separated by at most `gap` intervals belong to one episode"""
    a = np.asarray(alarm_idx); return 0 if len(a) == 0 else 1 + int((np.diff(a) > gap).sum())
def episodes_at(sets, tau, gap=GAP): return sum(episodes(np.flatnonzero((s['score'] > tau) & s['elig'] & s['clean']), gap) for s in sets)
def find_tau(sets, per_set_burden, gap=GAP, npts=240):
    """lowest threshold whose number of CLEAN episodes is at most per_set_burden x number of sets (one set = one symbol-day).
    The episode count is not monotone when the threshold gets very low (episodes merge), so the threshold is lowered step by step
    (tail probabilities from 1e-6 to 0.5 on a log grid) and the search stops at the first step that exceeds the budget."""
    target = per_set_burden * len(sets); vals = np.concatenate([s['score'][s['elig'] & s['clean']] for s in sets])
    best = float(vals.max()) + 1.0
    for p in np.logspace(-6, np.log10(0.5), npts):
        tau = float(np.quantile(vals, 1.0 - p))
        if episodes_at(sets, tau, gap) <= target: best = tau
        else: break
    return best
def detect(sets, tau):
    """one row per event: detected if any eligible alarm interval lies in [start, end+1]; delay in seconds from the start"""
    rows = []
    for s in sets:
        al = (s['score'] > tau) & s['elig']
        for e in s['events']:
            seg = np.flatnonzero(al[e['start']:e['end'] + 2]); rows.append({'set': s['key'], 'pattern': e['pattern_name'], 'mag': e['mag_name'], 'detected': bool(len(seg)), 'delay_s': float(seg[0]) / 10.0 if len(seg) else np.nan})
    return pd.DataFrame(rows)
def summarize(df):
    t = df.groupby(['pattern', 'mag']).agg(events=('detected', 'size'), recall=('detected', 'mean'), median_delay_s=('delay_s', 'median')).reset_index(); o = {'events': len(df), 'recall': float(df['detected'].mean()), 'median_delay_s': float(df['delay_s'].median())}; return t, o

# ---------------------------------------------------------------- tests
def unit_tests(tmp):
    import stage_e_lib as E
    tmp = Path(tmp); out = []
    def add(i, nm, ok, det=''): out.append((i, nm, bool(ok), str(det)))
    add('J01', 'episodes by hand (gap rule)', episodes([5, 6, 7]) == 1 and episodes([5, 200]) == 2 and episodes([5, 105, 300]) == 2 and episodes([]) == 0)
    nodes, glob, flags, sc, q = E.make_toy(n=40000); n = len(flags); rng = np.random.default_rng(3)
    d = node_abs_change(nodes); add('J02', 'node change by hand (row 0 is zero, row i = |x_i - x_(i-1)|)', np.all(d[0] == 0) and abs(float(d[5, 7]) - abs(float(nodes[5, 7, LOGQ]) - float(nodes[4, 7, LOGQ]))) < 1e-5)
    # causality: changing the future leaves earlier scores identical
    mae = np.full(NN, 0.1); s1 = score_persistence(nodes, mae)[0]; s1z = score_activity_z(nodes); n2 = nodes.copy(); n2[30000:, :, LOGQ] += 5.0; s2 = score_persistence(n2, mae)[0]; s2z = score_activity_z(n2)
    add('J03', 'scores are causal: a change after interval 30000 does not alter any earlier score', np.array_equal(s1[:30000], s2[:30000]) and np.array_equal(s1z[:30000], s2z[:30000]) and not np.array_equal(s1[30000:], s2[30000:]))
    ev = E.place_events(flags, None, 1, rng, margin=100, gap=100); nn, gg, pa, ma, evl = E.inject_day(nodes, glob, flags, sc, ev, 7)
    win = np.arange(700, n - 2, 1).astype(np.int32); win = win[(flags[win] == 0) & (flags[win + 1] == 0)]; elig = eligible_targets(win, n); cl = clean_mask(n, evl)
    mae = train_node_mae([_toy_dir(tmp, nodes, win)], [flags]); st, sa = score_persistence(nn, mae); S1 = {'key': 'toy', 'score': st, 'elig': elig, 'clean': cl, 'events': evl}
    tau = find_tau([S1], 2.0); epi = episodes_at([S1], tau)
    add('J04', 'the threshold search gives at most the requested number of clean episodes (and is not far below it)', epi <= 2 and tau > 0, f'episodes {epi} tau {tau:.2f}')
    df = detect([S1], find_tau([S1], 20.0)); t, o = summarize(df); easy = df[df['mag'] == 'easy']['detected'].mean(); sub = df[df['mag'] == 'subtle']['detected'].mean()
    add('J05', 'on toy data the persistence score finds easy events and finds more easy than subtle ones', easy >= 0.75 and easy >= sub, f'easy {easy:.2f} subtle {sub:.2f} overall {o["recall"]:.2f}')
    # a score of pure noise should give recall near the share of alarm time (sanity: no label leakage)
    S0 = dict(S1); S0['score'] = rng.normal(size=n).astype(np.float32); d0 = detect([S0], find_tau([S0], 5.0)); add('J06', 'a random score does not find events reliably (no leakage through the evaluation)', d0['detected'].mean() < 0.6, f'recall {d0["detected"].mean():.2f}')
    return out
def _toy_dir(tmp, nodes, win):
    d = Path(tmp) / 'toy_train'; d.mkdir(parents=True, exist_ok=True); np.save(d / 'nodes.npy', nodes); np.save(d / 'win_end.npy', win); return d

Writing stage_e2_lib.py


In [ ]:
%%writefile stage_f2_lib.py
"""TradeSense C03 - Stage F2 library: supervised order-book manipulation DETECTION on labelled synthetic injections.
Detectors: DetTGNN (M1 with chain/spread/global edges, M0 with global links only), DetMLP (B3d), hand-made window features + XGBoost (B6).
Label of the window ending at interval t = pattern id of the event running at t (0 = none). Windows in the 60 intervals after an event are ambiguous and ignored in training.
Uses the frozen D1 library (staged_lib) and the D2 graph code (staged_d2_lib) unchanged. Test days are refused."""
import os, json, time
from pathlib import Path
import numpy as np
import torch, torch.nn as nn
import staged_lib as S
import staged_d2_lib as D2
from staged_lib import SEQ, NN, NCH, GCH, LOGQ
NCLS = 5; IGN = 60; DLQ = 3

class DetBank:
    def __init__(self, root, keys, device='cpu', chunk=100_000, stride=1):
        self.dir = Path(root) / 'sets'; self.keys = list(keys); self.device = device; self.summ = {}; total = 0
        for k in self.keys:
            s = json.loads((self.dir / k / 'set_summary.json').read_text()); self.summ[k] = s
            if s['split'] == 'test': raise PermissionError(f'{k} is a TEST set: refused')
            assert s.get('injected'), f'{k} is not an injected set'; total += int(s['n_intervals'])
        self.N = total; self.nodes = torch.empty((total, NN, NCH), dtype=torch.float16, device=device); self.glob = torch.empty((total, GCH), dtype=torch.float16, device=device)
        self.lab = np.zeros(total, np.int8); self.ign = np.zeros(total, bool); self.elig = np.zeros(total, bool); self.off = {}; self.events = []; o = 0
        for i, k in enumerate(self.keys):
            arr = np.load(self.dir / k / 'nodes.npy', mmap_mode='r'); g = np.load(self.dir / k / 'glob.npy', mmap_mode='r'); n = arr.shape[0]; assert arr.shape[1:] == (NN, NCH) and g.shape == (n, GCH)
            for a in range(0, n, chunk): b = min(a + chunk, n); self.nodes[o + a:o + b] = torch.from_numpy(np.array(arr[a:b])).to(device)
            self.glob[o:o + n] = torch.from_numpy(np.array(g)).to(device); self.lab[o:o + n] = np.load(self.dir / k / 'event_pattern.npy'); w = np.load(self.dir / k / 'win_end.npy').astype(np.int64)
            w = w[(w % stride) == 0]; self.elig[o + w] = True
            for e in json.loads((self.dir / k / 'events.json').read_text()):
                self.ign[o + e['end'] + 1:min(o + e['end'] + 1 + IGN, o + n)] = True; self.events.append({**e, 'set': k, 'g_start': o + e['start'], 'g_end': o + e['end']})
            self.off[k] = (o, n); o += n; del arr, g
        self.ar = torch.arange(-SEQ + 1, 1, device=device)
    def batch_x(self, g):
        idx = g[:, None] + self.ar; return self.nodes[idx].float(), self.glob[idx].float()
    def pools(self):
        pos = np.flatnonzero(self.elig & (self.lab > 0)); neg = np.flatnonzero(self.elig & (self.lab == 0) & ~self.ign); return pos, neg
    def interval_activity(self, chunk=200_000):
        A = np.zeros(self.N, np.float32)
        for a in range(0, self.N, chunk):
            b = min(a + chunk, self.N); lo = max(a - 1, 0); x = self.nodes[lo:b, :, LOGQ].float(); d = torch.zeros_like(x); d[1:] = (x[1:] - x[:-1]).abs(); A[a:b] = d[(a - lo):].sum(1).cpu().numpy()
        for k, (o, n) in self.off.items(): A[o] = 0
        return A
    def window_activity(self, steps=10):
        cs = np.cumsum(self.interval_activity(), dtype=np.float64); act = np.zeros(self.N, np.float32); act[steps:] = (cs[steps:] - cs[:-steps]); return act

# ---------------------------------------------------------------- models
class DetTGNN(D2.TGNN):
    def __init__(s, h=64, edges=D2.EDGES_M1, layers=2):
        super().__init__(h, edges, layers); del s.head_n, s.head_g; s.head = nn.Sequential(nn.Linear(6 * h, 2 * h), nn.ReLU(), nn.Linear(2 * h, NCLS))
    def forward(s, xn, xg, last=None):
        B, T = xn.shape[0], xn.shape[1]; em = s.emb.weight; NG = D2.NG
        n_in = torch.cat([xn, em[:NN][None, None].expand(B, T, -1, -1)], -1); g_in = torch.cat([xg[:, :, None, :], em[NN][None, None, None].expand(B, T, 1, -1)], -1)
        h = torch.cat([s.enc_n(n_in), s.enc_g(g_in)], 2)
        for l in range(s.layers):
            agg = s.self_lin[l](h)
            for e in s.edges: agg = agg + s.edge_lin[l][e](torch.matmul(getattr(s, 'A_' + e).to(h.dtype), h))
            h = h + torch.relu(s.norm[l](agg))
        seq = h.permute(0, 2, 1, 3).reshape(B * NG, T, s.h); out, _ = s.gru(seq); feat = torch.cat([out[:, -1], h[:, -1].reshape(B * NG, s.h)], 1).reshape(B, NG, 2 * s.h)
        nodes_f = feat[:, :NN]; pooled = torch.cat([nodes_f.mean(1), nodes_f.amax(1), feat[:, NN]], 1); return s.head(pooled).float()
def make_det_m1(cfg): return DetTGNN(cfg['hidden'], D2.EDGES_M1)
def make_det_m0(cfg): return DetTGNN(cfg['hidden'], D2.EDGES_M0)
class DetMLP(nn.Module):
    def __init__(s, h):
        super().__init__(); s.net = nn.Sequential(nn.Linear(SEQ * NN * NCH + SEQ * GCH, h), nn.ReLU(), nn.Linear(h, h), nn.ReLU(), nn.Linear(h, NCLS))
    def forward(s, xn, xg, last=None): return s.net(torch.cat([xn.flatten(1), xg.flatten(1)], 1)).float()
def make_det_mlp(cfg): return DetMLP(cfg['hidden'])

# ---------------------------------------------------------------- hand-made window features for B6
RANKS = list(range(10)) + [25 + i for i in range(10)]
def b6_features(xn, xg):
    """xn (B,50,50,6), xg (B,50,3) -> (B,149) float32. All features use the 50 intervals of the window only (causal)."""
    lq = xn[..., LOGQ][:, :, RANKS]; dl = xn[..., DLQ][:, :, RANKS]; adl = dl.abs(); base = lq.median(1).values; dev = lq - base[:, None]
    f = [adl[:, -10:].sum(1), adl.sum(1), dev.amax(1), lq[:, -1] - base, dev.amin(1)]
    nz = (adl > 0.05); sg = torch.sign(dl) * nz; flips = ((sg[:, 1:] * sg[:, :-1]) < 0).sum(1).float(); f += [flips, nz.sum(1).float()]
    ig = xg[:, :, 1]; f.append(torch.stack([ig[:, -1], ig.mean(1), ig.std(1), (ig[:, 1:] - ig[:, :-1]).abs().amax(1)], 1))
    a_all = xn[..., DLQ].abs().sum(2); f.append(torch.stack([a_all[:, -5:].sum(1), a_all[:, -10:].sum(1), a_all[:, -25:].sum(1), a_all.sum(1)], 1)); f.append(xn[..., DLQ].abs()[:, -5:].amax((1, 2))[:, None])
    return torch.cat(f, 1).float()

# ---------------------------------------------------------------- scoring helpers
def det_score_from_logits(lg): return torch.logsumexp(lg[:, 1:], 1) - lg[:, 0]
@torch.no_grad()
def score_windows(model, bank, gidx, device, amp=False, bs=512):
    model.eval(); out = np.zeros(len(gidx), np.float32)
    for a in range(0, len(gidx), bs):
        g = torch.from_numpy(gidx[a:a + bs]).to(device); xn, xg = bank.batch_x(g)
        with D2.amp_ctx(device, amp): lg = model(xn, xg)
        out[a:a + len(g)] = det_score_from_logits(lg.float()).cpu().numpy()
    return out
@torch.no_grad()
def feature_matrix(bank, gidx, device, bs=1024):
    out = []
    for a in range(0, len(gidx), bs):
        g = torch.from_numpy(gidx[a:a + bs]).to(device); xn, xg = bank.batch_x(g); out.append(b6_features(xn, xg).cpu().numpy())
    return np.concatenate(out) if out else np.zeros((0, 149), np.float32)
def recall_at_fpr(pos_scores, neg_scores, fpr=1e-3):
    k = max(int(np.ceil(fpr * len(neg_scores))), 1); thr = np.sort(neg_scores)[-k]; return float((np.asarray(pos_scores) > thr).mean())
def val_subset(bank, n_pos=20000, n_neg=100000, seed=777):
    rng = np.random.default_rng(seed); pos, neg = bank.pools(); pos = np.sort(rng.choice(pos, min(n_pos, len(pos)), replace=False)); neg = np.sort(rng.choice(neg, min(n_neg, len(neg)), replace=False)); return pos, neg
def sample_epoch(rng, pos, neg, hard_neg, P, neg_mult, hard_share=0.5):
    p = rng.choice(pos, min(P, len(pos)), replace=False); nn_ = neg_mult * len(p); nh = int(nn_ * hard_share); h = rng.choice(hard_neg, min(nh, len(hard_neg)), replace=False) if len(hard_neg) else np.zeros(0, np.int64)
    u = rng.choice(neg, nn_ - len(h), replace=False); idx = np.concatenate([p, h, u]); rng.shuffle(idx); return idx

# ---------------------------------------------------------------- resumable trainer
def fit_det(make_model, cfg, bank_tr, vs_bank, vs, pools, out_dir, seed, device='cpu', amp=False, deadline=None, max_epochs=10, patience=3, bs=256, P=20000, neg_mult=4, log=print, _halt_after=None):
    out = Path(out_dir); out.mkdir(parents=True, exist_ok=True)
    if (out / 'done.json').exists(): return json.loads((out / 'done.json').read_text())
    assert all(bank_tr.summ[k]['split'] == 'train' for k in bank_tr.keys), 'training bank contains a non-training set'
    pos, neg, hard = pools; torch.manual_seed(seed); model = make_model(cfg).to(device); opt = torch.optim.AdamW(model.parameters(), lr=cfg['lr'], weight_decay=1e-4); scaler = torch.amp.GradScaler('cuda', enabled=bool(amp and device == 'cuda'))
    state = {'epoch': 0, 'best': -1.0, 'best_epoch': 0, 'bad': 0, 'hist': []}
    for p in sorted(out.glob('resume_e*.pt'), reverse=True):
        try: ck = torch.load(p, map_location='cpu', weights_only=False)
        except Exception: continue
        model.load_state_dict(ck['model']); opt.load_state_dict(ck['opt']); scaler.load_state_dict(ck['scaler']); state = ck['state']; log(f'  resumed from {p.name} (epoch {state["epoch"]})'); break
    lossf = nn.CrossEntropyLoss()
    while state['epoch'] < max_epochs and state['bad'] < patience:
        e = state['epoch'] + 1; t0 = time.time(); rng = np.random.default_rng([seed, e]); idx = sample_epoch(rng, pos, neg, hard, P, neg_mult); model.train(); tot = 0.0; cnt = 0
        for a in range(0, len(idx), bs):
            if deadline is not None and time.time() > deadline: log('  time budget reached: stop here (this epoch is redone on resume), run again to continue'); return None
            gi = idx[a:a + bs]; g = torch.from_numpy(gi).to(device); y = torch.from_numpy(bank_tr.lab[gi].astype(np.int64)).to(device); xn, xg = bank_tr.batch_x(g)
            with D2.amp_ctx(device, amp): lg = model(xn, xg)
            loss = lossf(lg.float(), y); opt.zero_grad(set_to_none=True); scaler.scale(loss).backward(); scaler.unscale_(opt); torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0); scaler.step(opt); scaler.update(); tot += loss.item() * len(gi); cnt += len(gi)
        t_train = time.time() - t0; ps = score_windows(model, vs_bank, vs[0], device, amp); ns = score_windows(model, vs_bank, vs[1], device, amp); val = recall_at_fpr(ps, ns); torch.save(model.state_dict(), out / f'weights_e{e:03d}.pt')
        state['epoch'] = e; state['hist'].append({'epoch': e, 'train_loss': tot / max(cnt, 1), 'val_recall_at_fpr1e-3': val, 'train_seconds': round(t_train, 1), 'epoch_seconds': round(time.time() - t0, 1)})
        if val > state['best'] + 1e-9: state['best'] = val; state['best_epoch'] = e; state['bad'] = 0
        else: state['bad'] += 1
        tmp = out / f'resume_e{e:03d}.pt.tmp'; torch.save({'model': model.state_dict(), 'opt': opt.state_dict(), 'scaler': scaler.state_dict(), 'state': state}, tmp); os.replace(tmp, out / f'resume_e{e:03d}.pt')
        for p in out.glob('resume_e*.pt'):
            if int(p.stem[8:]) < e - 1: p.unlink()
        with open(out / 'log.jsonl', 'a') as f: f.write(json.dumps(state['hist'][-1]) + '\n')
        log(f'  epoch {e}: train loss {state["hist"][-1]["train_loss"]:.4f} | val recall@FPR1e-3 {val:.3f} | best {state["best"]:.3f} (e{state["best_epoch"]}) | {state["hist"][-1]["epoch_seconds"]} s')
        if _halt_after is not None and e >= _halt_after: return None
    done = {'cfg': cfg, 'seed': seed, 'best_val': state['best'], 'best_epoch': state['best_epoch'], 'epochs_run': state['epoch'], 'history': state['hist'], 'n_params': int(sum(p.numel() for p in model.parameters())), 'mean_epoch_seconds': float(np.mean([h['epoch_seconds'] for h in state['hist']])), 'P': P, 'neg_mult': neg_mult, 'batch_size': bs, 'max_epochs': max_epochs, 'patience': patience, 'amp': bool(amp and device == 'cuda')}
    S.dump_json(done, out / 'done.json'); return done
def load_best_det(make_model, cfg, out_dir, device='cpu'):
    d = json.loads((Path(out_dir) / 'done.json').read_text()); m = make_model(cfg).to(device); m.load_state_dict(torch.load(Path(out_dir) / f'weights_e{d["best_epoch"]:03d}.pt', map_location=device)); m.eval(); return m, d

# ---------------------------------------------------------------- B6 (XGBoost on hand-made features)
def train_b6(bank_tr, pools, vs_bank, vs, device, out_path, seed=0, n_pos=60000, neg_mult=4, rounds=400, log=print):
    import xgboost as xgb
    pos, neg, hard = pools; rng = np.random.default_rng([seed, 99]); idx = sample_epoch(rng, pos, neg, hard, n_pos, neg_mult); X = feature_matrix(bank_tr, idx, device); y = bank_tr.lab[idx].astype(np.int64)
    Xv = np.concatenate([feature_matrix(vs_bank, vs[0], device), feature_matrix(vs_bank, vs[1], device)]); yv = np.concatenate([vs_bank.lab[vs[0]], vs_bank.lab[vs[1]]]).astype(np.int64)
    dev = 'cuda' if (device == 'cuda') else 'cpu'
    clf = xgb.XGBClassifier(n_estimators=rounds, max_depth=6, learning_rate=0.08, subsample=0.8, colsample_bytree=0.8, objective='multi:softprob', num_class=NCLS, tree_method='hist', device=dev, eval_metric='mlogloss', early_stopping_rounds=30, random_state=seed, n_jobs=4)
    clf.fit(X, y, eval_set=[(Xv, yv)], verbose=False); clf.save_model(str(out_path)); ps = b6_scores(clf, vs_bank, vs[0], device); ns = b6_scores(clf, vs_bank, vs[1], device)
    return clf, {'best_iteration': int(clf.best_iteration), 'val_recall_at_fpr1e-3': recall_at_fpr(ps, ns), 'n_train': int(len(idx))}
def b6_scores(clf, bank, gidx, device, bs=100_000):
    out = np.zeros(len(gidx), np.float32)
    for a in range(0, len(gidx), bs):
        X = feature_matrix(bank, gidx[a:a + bs], device); p = clf.predict_proba(X); out[a:a + len(X)] = np.log(np.clip(1 - p[:, 0], 1e-7, 1)) - np.log(np.clip(p[:, 0], 1e-7, 1))
    return out
def load_b6(path):
    import xgboost as xgb
    c = xgb.XGBClassifier(); c.load_model(str(path)); return c

# ---------------------------------------------------------------- tests
def _toy_root(tmp, name='toy', split='train', scale=('choice', [1.5]), n_per_pair=2, n=20000, seed=3):
    import stage_e_lib as E, stage_f_lib as F
    nodes, glob, flags, sc, q = E.make_toy(n=n); d = Path(tmp) / f'{name}_src'; d.mkdir(parents=True, exist_ok=True); np.save(d / 'nodes.npy', nodes); np.save(d / 'glob.npy', glob); np.save(d / 'flags.npy', flags)
    np.save(d / 'win_end.npy', np.arange(700, n - 10, 1).astype(np.int32)); (d / 'set_summary.json').write_text(json.dumps({'split': split, 'n_intervals': n}))
    root = Path(tmp) / name; F.build_set_f(d, root / 'sets' / 'BTCUSDT_2024-05-01', sc, n_per_pair, seed, (1, 2, 3, 4), scale, margin=100, gap=100, allow_train=(split == 'train')); return root
def unit_tests(tmp):
    tmp = Path(tmp); out = []
    def add(i, nm, ok, det=''): out.append((i, nm, bool(ok), str(det)))
    root = _toy_root(tmp); bank = DetBank(root, ['BTCUSDT_2024-05-01']); ev = bank.events; pos, neg = bank.pools()
    ok1 = all(bank.lab[e['g_start']:e['g_end'] + 1].min() == e['pattern'] for e in ev) and not bank.ign[ev[0]['g_start']] and bank.ign[ev[0]['g_end'] + 1] and bank.ign[ev[0]['g_end'] + IGN] and not bank.ign[ev[0]['g_end'] + IGN + 1]
    add('M01', 'labels equal the event pattern inside events; ignore zone starts right after the event and lasts 60 intervals', ok1)
    ev_all = np.zeros(bank.N, bool)
    for e in ev: ev_all[e['g_start']:e['g_end'] + 1] = True
    add('M02', 'positive pool lies inside events, negative pool never inside an event or its ignore zone', ev_all[pos].all() and not ev_all[neg].any() and not bank.ign[neg].any() and len(pos) == int((ev_all & bank.elig).sum()))
    for sp in ('test',):
        d = Path(tmp) / 'tt' / 'sets' / 'X'; d.mkdir(parents=True); (d / 'set_summary.json').write_text(json.dumps({'split': sp, 'n_intervals': 10, 'injected': True}))
        try: DetBank(Path(tmp) / 'tt', ['X']); okr = False
        except PermissionError: okr = True
    add('M03', 'a test set is refused by the loader', okr)
    g = torch.from_numpy(pos[:8]); xn, xg = bank.batch_x(g); torch.manual_seed(0); m1 = make_det_m1({'hidden': 16}); m0 = make_det_m0({'hidden': 16}); o1 = m1(xn, xg); o0 = m0(xn, xg)
    add('M04', 'both graph detectors give (batch, 5) logits; chain/spread edges exist in M1 only', o1.shape == (8, NCLS) and o0.shape == (8, NCLS) and any('chain' in k for k in m1.state_dict()) and not any('chain' in k for k in m0.state_dict() if 'edge_lin' in k) and torch.isfinite(o1).all())
    # B6 by hand: a step of height 2 on one rank (rank 3 of the bid side) starting at window position 30
    xn0 = torch.zeros(1, SEQ, NN, NCH); xg0 = torch.zeros(1, SEQ, GCH); xn0[0, 30:, 3, LOGQ] = 2.0; xn0[0, 30, 3, DLQ] = 2.0; fe = b6_features(xn0, xg0)[0]
    add('M05', 'B6 features by hand: deviation from window median, total change, flips, number of changes, total activity', abs(float(fe[43]) - 2.0) < 1e-6 and abs(float(fe[23]) - 2.0) < 1e-6 and abs(float(fe[63]) - 2.0) < 1e-6 and float(fe[103]) == 0.0 and float(fe[123]) == 1.0 and float(fe[148]) == 0.0 and abs(float(fe[147]) - 2.0) < 1e-6 and len(fe) == 149)
    add('M06', 'recall at a false-alarm rate by hand', abs(recall_at_fpr([95, 96, 97, 3], list(range(100)), 0.05) - 0.5) < 1e-9 and recall_at_fpr([1, 2], list(range(100)), 0.05) == 0.0)
    rv = _toy_root(tmp, 'toyv', 'val', ('choice', [1.5]), 2, 20000, seed=11); bv = DetBank(rv, ['BTCUSDT_2024-05-01']); vs = val_subset(bv, 5000, 5000)
    act = bank.window_activity(10); hard = neg[act[neg] >= np.quantile(act[neg], 0.8)]; pools = (pos, neg, hard)
    add('M07', 'hard negatives are the busiest 20% of normal windows', len(hard) > 0 and act[hard].min() >= np.quantile(act[neg], 0.8) - 1e-6 and act[hard].mean() > act[neg].mean())
    logs = []; cfg = {'hidden': 32, 'lr': 2e-3}; d = fit_det(make_det_mlp, cfg, bank, bv, vs, pools, tmp / 'fit_mlp', 1, 'cpu', max_epochs=6, patience=6, bs=128, P=1000, neg_mult=3, log=logs.append); m, dd = load_best_det(make_det_mlp, cfg, tmp / 'fit_mlp')
    ps = score_windows(m, bv, vs[0], 'cpu'); ns = score_windows(m, bv, vs[1], 'cpu'); rc = recall_at_fpr(ps, ns, 0.01); add('M08', 'the MLP detector learns strong toy events (validation recall at 1% false-alarm rate above 0.25, chance is 0.01)', rc > 0.25, f'recall {rc:.2f} (chance 0.01)')
    cfg1 = {'hidden': 16, 'lr': 2e-3}; dd1 = fit_det(make_det_m1, cfg1, bank, bv, vs, pools, tmp / 'fit_m1', 2, 'cpu', max_epochs=3, patience=3, bs=128, P=300, neg_mult=2, log=logs.append); m1_, _ = load_best_det(make_det_m1, cfg1, tmp / 'fit_m1')
    ps1 = score_windows(m1_, bv, vs[0], 'cpu'); ns1 = score_windows(m1_, bv, vs[1], 'cpu'); add('M09', 'the graph detector trains without error and its scores are finite and above chance on toy events', np.isfinite(ps1).all() and float(ps1.mean()) > float(ns1.mean()), f'mean pos {ps1.mean():.2f} neg {ns1.mean():.2f}')
    fit_det(make_det_mlp, cfg, bank, bv, vs, pools, tmp / 'full', 5, 'cpu', max_epochs=2, patience=5, bs=128, P=600, neg_mult=3, log=lambda *a: None)
    fit_det(make_det_mlp, cfg, bank, bv, vs, pools, tmp / 'half2', 5, 'cpu', max_epochs=2, patience=5, bs=128, P=600, neg_mult=3, log=lambda *a: None, _halt_after=1); fit_det(make_det_mlp, cfg, bank, bv, vs, pools, tmp / 'half2', 5, 'cpu', max_epochs=2, patience=5, bs=128, P=600, neg_mult=3, log=lambda *a: None)
    wa = torch.load(tmp / 'full' / 'weights_e002.pt'); wb = torch.load(tmp / 'half2' / 'weights_e002.pt'); add('M10', 'stopping after one epoch and resuming gives the same weights as an uninterrupted run', all(torch.equal(wa[k], wb[k]) for k in wa))
    a1 = score_windows(m, bv, vs[0], 'cpu', bs=64); a2 = score_windows(m, bv, vs[0], 'cpu', bs=1000); add('M11', 'scores do not depend on the batch size', np.allclose(a1, a2, atol=1e-5))
    try:
        clf, info = train_b6(bank, pools, bv, vs, 'cpu', tmp / 'b6.json', rounds=60); pb = b6_scores(clf, bv, vs[0], 'cpu'); nb = b6_scores(clf, bv, vs[1], 'cpu'); rb = recall_at_fpr(pb, nb, 0.01); add('M12', 'B6 (hand-made features + XGBoost) learns toy events (recall at 1% false alarms above 0.15) and reloads identically', rb > 0.15 and np.allclose(b6_scores(load_b6(tmp / 'b6.json'), bv, vs[0], 'cpu'), pb, atol=1e-5), f'recall {rb:.2f}')
    except Exception as ex: add('M12', 'B6 trains', False, repr(ex)[:200])
    return out

Writing stage_f2_lib.py


## 3 · Safety checks: GPU, E1/F1 finished, library hash equal to D1, frozen configuration

In [ ]:
import torch
sys.path.insert(0, str(Path.cwd()))
import staged_lib as S, staged_d2_lib as D2, stage_e_lib as E, stage_f_lib as F, stage_e2_lib as L2, stage_f2_lib as L
def _sha(p): return hashlib.sha256(Path(p).read_bytes()).hexdigest()
LIBS = {n: _sha(n + '.py') for n in ['staged_lib', 'staged_d2_lib', 'stage_e_lib', 'stage_f_lib', 'stage_e2_lib', 'stage_f2_lib']}
DEV = 'cuda' if torch.cuda.is_available() else 'cpu'; GPU = torch.cuda.get_device_name(0) if DEV == 'cuda' else None
print('device', DEV, GPU or '(no GPU: this will be very slow; switch the runtime to T4)')
SC_DIR = BASE / STAGEC_NAME; EDIR = BASE / E_NAME; FDIR = BASE / F_NAME; D1DIR = BASE / D1_NAME
rep_d1 = json.loads((D1DIR / 'reports' / 'd1_report.json').read_text()); assert rep_d1['library_sha256'][:12] == LIBS['staged_lib'][:12], 'staged_lib.py differs from the one used in D1'
assert json.loads((EDIR / 'reports' / 'e1_report.json').read_text())['verdict'] == 'E1 COMPLETE', 'E1 is not complete'
assert json.loads((FDIR / 'reports' / 'f1_report.json').read_text())['verdict'] == 'F1 COMPLETE', 'F1 is not complete'
TKEYS = [f'{s}_{d}' for d in TRAIN_DAYS for s in SYMS]; VKEYS = [f'{s}_{d}' for d in VAL_DAYS for s in SYMS]
OUT = FDIR / 'f2'; (OUT / 'config').mkdir(parents=True, exist_ok=True); (OUT / 'runs').mkdir(exist_ok=True); (OUT / 'scores').mkdir(exist_ok=True); (OUT / 'reports').mkdir(exist_ok=True)
FROZEN = {'cfg': CFG, 'max_epochs': MAX_EPOCHS, 'patience': PATIENCE, 'batch': BATCH, 'p_pos': P_POS, 'neg_mult': NEG_MULT, 'amp': AMP, 'seeds': SEEDS, 'stride': STRIDE, 'burdens': BURDENS, 'primary': PRIMARY, 'n_boot': N_BOOT, 'min_gain': MIN_GAIN, 'train': TKEYS, 'val': VKEYS, 'ign': L.IGN, 'protocol': 'claude/protocol_v3_stageF.md'}
def _canon(o): return hashlib.sha256(json.dumps(o, sort_keys=True).encode()).hexdigest()
fp = OUT / 'config' / 'frozen_config_f2.json'
if not fp.exists(): fp.write_text(json.dumps({'config': FROZEN, 'library_sha256': LIBS, 'sha256': _canon(FROZEN)}, indent=1)); print('frozen config written')
else: assert json.loads(fp.read_text())['sha256'] == _canon(FROZEN), 'settings differ from the saved frozen configuration; do not change them while resuming'; print('frozen config matches')
assert not (D1DIR / 'config' / 'freeze_d3.json').exists() or True
print('libraries', {k: v[:8] for k, v in LIBS.items()})

device cuda Tesla T4
frozen config written
libraries {'staged_lib': '85e7c1f4', 'staged_d2_lib': '54b345bc', 'stage_e_lib': 'a67aa38f', 'stage_f_lib': '375cb175', 'stage_e2_lib': '776e7143', 'stage_f2_lib': '324ca678'}


## 4 · Tests (injector, E2 scoring helpers, Stage F builders, detector library)

In [ ]:
import tempfile
with tempfile.TemporaryDirectory() as t_: ER = F.unit_tests(t_) + L2.unit_tests(t_) + L.unit_tests(t_)
display(pd.DataFrame(ER, columns=['id', 'test', 'pass', 'detail'])); assert all(r[2] for r in ER), 'A test failed: stop here and send me this table.'
print(f'tests: {sum(r[2] for r in ER)}/{len(ER)} passed')

,id,test,pass,detail
0,L01,placement gives 12 events with a scale each; d...,True,
1,L02,duration multiplier 2 makes events longer on a...,True,
2,L03,spoof with scale 2 adds about twice the extra ...,True,ratio 2.00
3,L04,flicker switches on and off several times and ...,True,23 switches
4,L05,nothing outside the event changes,True,
5,L06,test days refused; training needs allow_train;...,True,"{'test{}': True, 'train{}': True, ""val{'allow_..."
6,L07,label array marks exactly the event intervals ...,True,
7,L08,same seed gives identical sets; shifted valida...,True,
8,J01,episodes by hand (gap rule),True,
9,J02,"node change by hand (row 0 is zero, row i = |x...",True,


tests: 26/26 passed


## 5 · Load the training sets (injected May and June) and the two validation sets (July); test days stay closed

In [ ]:
t0 = time.time()
TRB = L.DetBank(FDIR / 'train_injected', TKEYS, device=DEV); VSTD = L.DetBank(EDIR / 'val_injected', VKEYS, device=DEV, stride=STRIDE); VSH = L.DetBank(FDIR / 'val_shift', VKEYS, device=DEV, stride=STRIDE)
assert all(TRB.summ[k]['split'] == 'train' for k in TKEYS) and all(VSTD.summ[k]['split'] == 'val' for k in VKEYS) and all(VSH.summ[k]['split'] == 'val' for k in VKEYS)
POS, NEG = TRB.pools(); ACT = TRB.window_activity(10); HARD = NEG[ACT[NEG] >= np.quantile(ACT[NEG], 0.8)]; POOLS = (POS, NEG, HARD); VS = L.val_subset(VSTD, 20000, 100000)
print(f'train windows: positive {len(POS):,}, normal {len(NEG):,} (hard {len(HARD):,}) | events {len(TRB.events)} | validation subset {len(VS[0]):,} + {len(VS[1]):,} | std val eligible {int(VSTD.elig.sum()):,} | shift val eligible {int(VSH.elig.sum()):,} | {time.time()-t0:.0f} s')
print('label shares in training windows:', {p: int((TRB.lab[POS] == p).sum()) for p in range(1, 5)})

train windows: positive 69,667, normal 3,311,367 (hard 662,274) | events 1200 | validation subset 7,039 + 100,000 | std val eligible 575,329 | shift val eligible 575,329 | 148 s
label shares in training windows: {1: 16681, 2: 24325, 3: 19806, 4: 8855}


## 6 · B6: hand-made features + XGBoost

In [ ]:
b6p = OUT / 'runs' / 'b6' / 'b6.json'; (OUT / 'runs' / 'b6').mkdir(parents=True, exist_ok=True)
if b6p.exists() and (OUT / 'runs' / 'b6' / 'info.json').exists(): B6 = L.load_b6(b6p); B6INFO = json.loads((OUT / 'runs' / 'b6' / 'info.json').read_text()); print('B6 already trained', B6INFO)
else:
    t0 = time.time(); B6, B6INFO = L.train_b6(TRB, POOLS, VSTD, VS, DEV, b6p); (OUT / 'runs' / 'b6' / 'info.json').write_text(json.dumps(B6INFO)); print('B6 trained', B6INFO, f'({time.time()-t0:.0f} s)')

B6 trained {'best_iteration': 399, 'val_recall_at_fpr1e-3': 0.2896718283847137, 'n_train': 300000} (35 s)


## 7 · Train M1, M0 and B3d (3 seeds each). Stops itself after `MAX_RUN_MINUTES`; run all again to continue

In [ ]:
DEADLINE = time.time() + MAX_RUN_MINUTES * 60; MAKE = {'M1': L.make_det_m1, 'M0': L.make_det_m0, 'B3d': L.make_det_mlp}; INFO = {}; PAUSED = False
for sd in SEEDS:
    for name in ['M1', 'M0', 'B3d']:
        if PAUSED: break
        tag = f'{name}_s{sd}'; print(tag)
        d = L.fit_det(MAKE[name], CFG, TRB, VSTD, VS, POOLS, OUT / 'runs' / tag, sd, DEV, AMP, DEADLINE, MAX_EPOCHS, PATIENCE, BATCH, P_POS, NEG_MULT)
        if d is None: PAUSED = True; break
        INFO[tag] = d; print(f'  done: best epoch {d["best_epoch"]}, val recall@FPR1e-3 {d["best_val"]:.3f}, {d["n_params"]:,} parameters, {d["mean_epoch_seconds"]:.0f} s/epoch')
if PAUSED: print('\nTime budget reached. Run all again (the same settings) to continue; finished runs are skipped.')
else: print('\nAll training runs finished.')

M1_s301
  epoch 1: train loss 0.5227 | val recall@FPR1e-3 0.309 | best 0.309 (e1) | 98.7 s
  epoch 2: train loss 0.3322 | val recall@FPR1e-3 0.386 | best 0.386 (e2) | 97.2 s
  epoch 3: train loss 0.2641 | val recall@FPR1e-3 0.377 | best 0.386 (e2) | 97.0 s
  epoch 4: train loss 0.2297 | val recall@FPR1e-3 0.420 | best 0.420 (e4) | 96.8 s
  epoch 5: train loss 0.1973 | val recall@FPR1e-3 0.423 | best 0.423 (e5) | 97.0 s
  epoch 6: train loss 0.1793 | val recall@FPR1e-3 0.460 | best 0.460 (e6) | 96.9 s
  epoch 7: train loss 0.1639 | val recall@FPR1e-3 0.424 | best 0.460 (e6) | 96.7 s
  epoch 8: train loss 0.1526 | val recall@FPR1e-3 0.405 | best 0.460 (e6) | 96.8 s
  epoch 9: train loss 0.1374 | val recall@FPR1e-3 0.443 | best 0.460 (e6) | 96.7 s
  done: best epoch 6, val recall@FPR1e-3 0.460, 110,173 parameters, 97 s/epoch
M0_s301
  epoch 1: train loss 0.5749 | val recall@FPR1e-3 0.292 | best 0.292 (e1) | 70.9 s
  epoch 2: train loss 0.3764 | val recall@FPR1e-3 0.298 | best 0.298 (e2) |

## 8 · Score the validation windows (every 3rd interval, seeds saved separately)

In [ ]:
if PAUSED: raise SystemExit('Training is not finished: run all again first.')
t0 = time.time(); GROUPS = {'std': VSTD, 'shift': VSH}; NAMES_NEURAL = ['M1', 'M0', 'B3d']
for gname, bank in GROUPS.items():
    gidx = np.flatnonzero(bank.elig); np.save(OUT / 'scores' / f'gidx_{gname}.npy', gidx)
    for name in NAMES_NEURAL:
        for sd in SEEDS:
            f = OUT / 'scores' / f'{name}_s{sd}_{gname}.npy'
            if f.exists(): continue
            m, _ = L.load_best_det(MAKE[name], CFG, OUT / 'runs' / f'{name}_s{sd}', DEV); np.save(f, L.score_windows(m, bank, gidx, DEV, AMP)); del m; torch.cuda.empty_cache() if DEV == 'cuda' else None; print(f'scored {name} s{sd} {gname} ({time.time()-t0:.0f} s)')
    f = OUT / 'scores' / f'B6_s0_{gname}.npy'
    if not f.exists(): np.save(f, L.b6_scores(B6, bank, gidx, DEV)); print(f'scored B6 {gname} ({time.time()-t0:.0f} s)')

scored M1 s301 std (162 s)
scored M1 s302 std (323 s)
scored M1 s303 std (486 s)
scored M0 s301 std (604 s)
scored M0 s302 std (722 s)
scored M0 s303 std (840 s)
scored B3d s301 std (841 s)
scored B3d s302 std (843 s)
scored B3d s303 std (845 s)
scored B6 std (852 s)
scored M1 s301 shift (1014 s)
scored M1 s302 shift (1176 s)
scored M1 s303 shift (1337 s)
scored M0 s301 shift (1455 s)
scored M0 s302 shift (1573 s)
scored M0 s303 shift (1691 s)
scored B3d s301 shift (1693 s)
scored B3d s302 shift (1694 s)
scored B3d s303 shift (1696 s)
scored B6 shift (1704 s)


## 9 · Untrained baseline scores on both validation sets

In [ ]:
MAEF = EDIR / 'e2' / 'scores' / 'train_node_mae.npy'
if MAEF.exists(): MAE = np.load(MAEF)
else:
    dirs = [SC_DIR / 'sets' / k for k in TKEYS]; MAE = L2.train_node_mae(dirs, [np.load(d / 'flags.npy') for d in dirs]); np.save(OUT / 'scores' / 'train_node_mae.npy', MAE)
BL = {}
for gname, root in (('std', EDIR / 'val_injected'), ('shift', FDIR / 'val_shift')):
    for k in VKEYS:
        f = OUT / 'scores' / f'untrained_{gname}_{k}.npz'
        if not f.exists():
            nodes = np.load(root / 'sets' / k / 'nodes.npy', mmap_mode='r'); st, sa = L2.score_persistence(nodes, MAE, 5); sz = L2.score_activity_z(nodes, 600); np.savez(f, b1_top5=st, b1_all=sa, b5=sz); print('untrained scores', gname, k)
        z = np.load(f); BL[(gname, k)] = {'B1-top5': z['b1_top5'], 'B1-all': z['b1_all'], 'B5': z['b5']}

untrained scores std BTCUSDT_2024-07-01
untrained scores std ETHUSDT_2024-07-01
untrained scores shift BTCUSDT_2024-07-01
untrained scores shift ETHUSDT_2024-07-01


## 10 · Thresholds at matched clean false-alarm burden, recall and delay

In [ ]:
MODELS = ['M1', 'M0', 'B3d', 'B6', 'B1-top5', 'B1-all', 'B5']; UNTRAINED = ['B1-top5', 'B1-all', 'B5']; TRAINED_NONGRAPH = ['B3d', 'B6']
def model_scores(name, gname, bank):
    """list of dicts (one per symbol-day) with full-length score arrays (seeds averaged per interval) for model `name`"""
    gidx = np.load(OUT / 'scores' / f'gidx_{gname}.npy'); res = {}
    if name in UNTRAINED: return {k: BL[(gname, k)][name] for k in VKEYS}
    if name == 'B6': arrs = [np.load(OUT / 'scores' / f'B6_s0_{gname}.npy')]
    else: arrs = [np.load(OUT / 'scores' / f'{name}_s{sd}_{gname}.npy') for sd in SEEDS]
    avg = np.mean(arrs, 0); full = np.full(bank.N, -1e9, np.float32); full[gidx] = avg
    for k in VKEYS: o, n = bank.off[k]; res[k] = full[o:o + n]
    return res
def make_sets(name, gname, bank):
    sc = model_scores(name, gname, bank); out = []
    for k in VKEYS:
        o, n = bank.off[k]; evl = [e for e in bank.events if e['set'] == k]; elig = bank.elig[o:o + n]
        out.append({'key': k, 'score': sc[k], 'elig': elig, 'clean': L2.clean_mask(n, evl, L2.NEAR), 'events': evl})
    return out
SETS = {(m, g): make_sets(m, g, b) for g, b in GROUPS.items() for m in MODELS}
TAU = {}; DET = {}; ROWS = []
def wilson(k, n, z=1.96):
    p = k / n; d = 1 + z * z / n; c = p + z * z / (2 * n); h = z * np.sqrt(p * (1 - p) / n + z * z / (4 * n * n)); return (c - h) / d, (c + h) / d
for (m, g), sets in SETS.items():
    for b in BURDENS:
        tau = L2.find_tau(sets, b, L2.GAP); TAU[(m, g, b)] = tau; df = L2.detect(sets, tau); DET[(m, g, b)] = df; lo, hi = wilson(int(df['detected'].sum()), len(df))
        ROWS.append({'model': m, 'set': g, 'burden/set': b, 'events': len(df), 'recall': df['detected'].mean(), 'CI': f'{lo:.2f}-{hi:.2f}', 'median delay s': df['delay_s'].median(), 'clean episodes (realised)': L2.episodes_at(sets, tau, L2.GAP) / len(sets)})
SUMMARY = pd.DataFrame(ROWS)
for g in ('std', 'shift'):
    print(f'=== recall, set = {g} ({"E1 standard" if g == "std" else "shifted: new pattern, longer events, other magnitudes"})'); display(SUMMARY[SUMMARY['set'] == g].pivot(index='model', columns='burden/set', values='recall').loc[MODELS].round(3))

=== recall, set = std (E1 standard)


burden/set,20,100,400
model,,,
M1,0.494,0.572,0.675
M0,0.433,0.514,0.611
B3d,0.319,0.364,0.467
B6,0.381,0.444,0.525
B1-top5,0.003,0.019,0.044
B1-all,0.000,0.003,0.042
B5,0.003,0.069,0.333


=== recall, set = shift (shifted: new pattern, longer events, other magnitudes)


burden/set,20,100,400
model,,,
M1,0.400,0.531,0.694
M0,0.358,0.431,0.550
B3d,0.236,0.294,0.433
B6,0.297,0.369,0.508
B1-top5,0.011,0.025,0.086
B1-all,0.008,0.019,0.072
B5,0.003,0.064,0.361


## 11 · Recall by pattern and magnitude; seed spread

In [ ]:
for g in ('std', 'shift'):
    for m in ['M1', 'M0', 'B3d', 'B6', 'B1-top5', 'B5']:
        df = DET[(m, g, PRIMARY)]
        if m in ('M1', 'B6', 'B5'):
            print(f'--- {m} on {g} at {PRIMARY} clean episodes per symbol-day: recall by pattern x magnitude'); display(df.groupby(['pattern', 'mag'])['detected'].mean().unstack()[['subtle', 'medium', 'easy']].round(2))
rows = []
for sd in SEEDS:
    for m in ('M1', 'M0', 'B3d'):
        arr = np.load(OUT / 'scores' / f'{m}_s{sd}_std.npy'); gidx = np.load(OUT / 'scores' / 'gidx_std.npy'); full = np.full(VSTD.N, -1e9, np.float32); full[gidx] = arr
        sets = [{'key': k, 'score': full[VSTD.off[k][0]:VSTD.off[k][0] + VSTD.off[k][1]], 'elig': VSTD.elig[VSTD.off[k][0]:VSTD.off[k][0] + VSTD.off[k][1]], 'clean': L2.clean_mask(VSTD.off[k][1], [e for e in VSTD.events if e['set'] == k], L2.NEAR), 'events': [e for e in VSTD.events if e['set'] == k]} for k in VKEYS]
        rows.append({'model': m, 'seed': sd, 'recall@primary (std)': L2.detect(sets, L2.find_tau(sets, PRIMARY, L2.GAP))['detected'].mean()})
SEEDTAB = pd.DataFrame(rows); print('single-seed recall at the primary burden (std set): the spread shows how much a seed matters'); display(SEEDTAB.pivot(index='seed', columns='model', values='recall@primary (std)').round(3))

--- M1 on std at 100 clean episodes per symbol-day: recall by pattern x magnitude


mag,subtle,medium,easy
pattern,,,
layering,0.33,0.87,0.97
liquidity_vacuum,0.00,0.17,0.33
quote_stuffing,1.00,1.00,1.00
spoof,0.13,0.33,0.73


--- B6 on std at 100 clean episodes per symbol-day: recall by pattern x magnitude


mag,subtle,medium,easy
pattern,,,
layering,0.13,0.47,0.67
liquidity_vacuum,0.13,0.27,0.53
quote_stuffing,1.00,1.00,1.00
spoof,0.03,0.03,0.07


--- B5 on std at 100 clean episodes per symbol-day: recall by pattern x magnitude


...

mag,subtle,medium,easy
pattern,,,
flicker,0.12,0.17,0.00
layering,0.17,0.42,0.71
liquidity_vacuum,0.25,0.21,0.38
quote_stuffing,1.00,1.00,1.00
spoof,0.04,0.04,0.04


--- B5 on shift at 100 clean episodes per symbol-day: recall by pattern x magnitude


mag,subtle,medium,easy
pattern,,,
flicker,0.12,0.08,0.08
layering,0.08,0.04,0.08
liquidity_vacuum,0.04,0.04,0.00
quote_stuffing,0.04,0.00,0.17
spoof,0.04,0.08,0.04


single-seed recall at the primary burden (std set): the spread shows how much a seed matters


model,B3d,M0,M1
seed,,,
301,0.350,0.458,0.589
302,0.333,0.492,0.553
303,0.361,0.442,0.469


## 12 · Pre-registered hypotheses H3a to H3d (event bootstrap)

In [ ]:
rng = np.random.default_rng(2024)
def boot(a, b, n=N_BOOT):
    a = np.asarray(a, float); b = np.asarray(b, float); idx = rng.integers(0, len(a), size=(n, len(a))); d = a[idx].mean(1) - b[idx].mean(1); return float(a.mean() - b.mean()), float(np.quantile(d, 0.025)), float(np.quantile(d, 0.975))
def best_of(cands, g, sel=None):
    r = {c: (DET[(c, g, PRIMARY)] if sel is None else DET[(c, g, PRIMARY)][sel(DET[(c, g, PRIMARY)])])['detected'].mean() for c in cands}; return max(r, key=r.get), r
def test(name_a, name_b, g, sel=None, burden=PRIMARY):
    da = DET[(name_a, g, burden)]; db = DET[(name_b, g, burden)]; assert (da['set'].values == db['set'].values).all() and (da['pattern'].values == db['pattern'].values).all()
    mask = np.ones(len(da), bool) if sel is None else sel(da).values; diff, lo, hi = boot(da['detected'].values[mask], db['detected'].values[mask]); return {'a': name_a, 'b': name_b, 'set': g, 'burden': burden, 'events': int(mask.sum()), 'recall_a': float(da['detected'].values[mask].mean()), 'recall_b': float(db['detected'].values[mask].mean()), 'diff': diff, 'ci_lo': lo, 'ci_hi': hi, 'holds': bool(diff >= MIN_GAIN and lo > 0)}
cmp_a, _ = best_of(TRAINED_NONGRAPH, 'std'); cmp_u_std, _ = best_of(UNTRAINED, 'std'); cmp_u_shift, _ = best_of(UNTRAINED, 'shift')
HY = {'H3a': test('M1', cmp_a, 'shift'), 'H3b': test('M1', cmp_u_std, 'std'), 'H3c': test('M1', cmp_u_shift, 'shift', sel=lambda d: d['pattern'] == 'flicker'), 'H3d': test('M1', 'M0', 'shift')}
HX = [test('M1', c, g, burden=b) for g in ('std', 'shift') for b in BURDENS for c in ['M0', 'B3d', 'B6', 'B5', 'B1-top5']]
table = pd.DataFrame([{'hypothesis': k, 'M1 vs': v['b'], 'set': v['set'], 'events': v['events'], 'recall M1': round(v['recall_a'], 3), 'recall other': round(v['recall_b'], 3), 'difference': round(v['diff'], 3), '95% CI': f'{v["ci_lo"]:.3f} to {v["ci_hi"]:.3f}', 'holds': v['holds']} for k, v in HY.items()])
print('Pre-registered hypotheses at', PRIMARY, 'clean episodes per symbol-day (comparators chosen by their recall on the standard set / the same set, never on the target events):'); display(table)
print('Extra comparisons (not pre-registered, all burdens):'); display(pd.DataFrame(HX).round(3))

Pre-registered hypotheses at 100 clean episodes per symbol-day (comparators chosen by their recall on the standard set / the same set, never on the target events):


,hypothesis,M1 vs,set,events,recall M1,recall other,difference,95% CI,holds
0,H3a,B6,shift,360,0.531,0.369,0.161,0.114 to 0.211,True
1,H3b,B5,std,360,0.572,0.069,0.503,0.444 to 0.561,True
2,H3c,B5,shift,72,0.236,0.097,0.139,0.014 to 0.264,True
3,H3d,M0,shift,360,0.531,0.431,0.100,0.064 to 0.139,True


Extra comparisons (not pre-registered, all burdens):


,a,b,set,burden,events,recall_a,recall_b,diff,ci_lo,ci_hi,holds
0,M1,M0,std,20,360,0.494,0.433,0.061,0.028,0.094,True
1,M1,B3d,std,20,360,0.494,0.319,0.175,0.136,0.217,True
2,M1,B6,std,20,360,0.494,0.381,0.114,0.067,0.158,True
3,M1,B5,std,20,360,0.494,0.003,0.492,0.442,0.544,True
4,M1,B1-top5,std,20,360,0.494,0.003,0.492,0.439,0.542,True
5,M1,M0,std,100,360,0.572,0.514,0.058,0.028,0.092,True
6,M1,B3d,std,100,360,0.572,0.364,0.208,0.164,0.253,True
7,M1,B6,std,100,360,0.572,0.444,0.128,0.081,0.178,True
8,M1,B5,std,100,360,0.572,0.069,0.503,0.444,0.561,True
9,M1,B1-top5,std,100,360,0.572,0.019,0.553,0.503,0.603,True


## 13 · Checks

In [ ]:
RT = []
def R(i, nm, ok, det=''): RT.append((i, nm, bool(ok), str(det)))
R('N01', 'no test day was opened: all banks hold train or validation sets only', all(TRB.summ[k]['split'] == 'train' for k in TKEYS) and all(b.summ[k]['split'] == 'val' for b in (VSTD, VSH) for k in VKEYS))
R('N02', 'training windows never come from validation days (disjoint sets)', set(TRB.keys).isdisjoint(set(VSTD.keys)))
R('N03', 'no flicker pattern in the training labels', int((TRB.lab == 5).sum()) == 0 and set(np.unique(TRB.lab)) <= {0, 1, 2, 3, 4})
R('N04', 'every model was scored on the same eligible windows (stride) and the same events', all(len(DET[(m, 'std', PRIMARY)]) == len(DET[('M1', 'std', PRIMARY)]) for m in MODELS) and all(len(DET[(m, 'shift', PRIMARY)]) == len(DET[('M1', 'shift', PRIMARY)]) for m in MODELS))
R('N05', 'realised clean episodes never exceed the requested burden (pooled)', all(r['clean episodes (realised)'] <= r['burden/set'] + 1e-9 for r in ROWS))
R('N06', 'recall does not fall when more false alarms are allowed', all(SUMMARY[(SUMMARY['model'] == m) & (SUMMARY['set'] == g)].sort_values('burden/set')['recall'].is_monotonic_increasing for m in MODELS for g in ('std', 'shift')))
R('N07', 'all scores on eligible windows are finite', all(np.isfinite(s['score'][s['elig']]).all() for sets in SETS.values() for s in sets))
R('N08', 'all training runs finished and none stopped at epoch 1', all(json.loads((OUT / 'runs' / f'{m}_s{sd}' / 'done.json').read_text())['epochs_run'] > 1 for m in ('M1', 'M0', 'B3d') for sd in SEEDS))
display(pd.DataFrame(RT, columns=['id', 'test', 'pass', 'detail']))

,id,test,pass,detail
0,N01,no test day was opened: all banks hold train o...,True,
1,N02,training windows never come from validation da...,True,
2,N03,no flicker pattern in the training labels,True,
3,N04,every model was scored on the same eligible wi...,True,
4,N05,realised clean episodes never exceed the reque...,True,
5,N06,recall does not fall when more false alarms ar...,True,
6,N07,all scores on eligible windows are finite,True,
7,N08,all training runs finished and none stopped at...,True,


## 14 · Saved report (never overwritten) and the SHARE BLOCK

In [ ]:
ALL = [(r[0], r[1], r[2], r[3]) for r in ER] + RT; ok = all(r[2] for r in ALL); VERDICT = 'F2 COMPLETE' if ok else 'F2 INCOMPLETE (a check failed)'
trainfo = {f'{m}_s{sd}': json.loads((OUT / 'runs' / f'{m}_s{sd}' / 'done.json').read_text()) for m in ('M1', 'M0', 'B3d') for sd in SEEDS}
rep = {'verdict': VERDICT, 'tests': ALL, 'frozen': FROZEN, 'library_sha256': LIBS, 'summary': SUMMARY.to_dict('records'), 'hypotheses': HY, 'extra': HX, 'comparators': {'H3a': cmp_a, 'H3b': cmp_u_std, 'H3c': cmp_u_shift}, 'b6': B6INFO, 'training': {k: {kk: v[kk] for kk in ('best_epoch', 'epochs_run', 'best_val', 'n_params', 'mean_epoch_seconds')} for k, v in trainfo.items()},
       'note': 'synthetic injection into real validation data; no test day opened; hypotheses evaluated on validation only and repeated in D3 on the test-day injected sets'}
rp = OUT / 'reports' / 'f2_report.json'
if not rp.exists(): rp.write_text(json.dumps(rep, indent=1, default=str))
pv = SUMMARY.pivot_table(index=['set', 'model'], columns='burden/set', values='recall')
Ls = ['==== F2 SHARE BLOCK (copy everything between the two ==== lines) ====', f'{VERDICT} | tests passed {sum(r[2] for r in ALL)}/{len(ALL)} | test days opened: NO | {GPU or "CPU"} | libs ' + ' '.join(v[:6] for v in LIBS.values())]
for g in ('std', 'shift'):
    for m in MODELS: Ls.append(f'{g:5s} {m:8s} recall @{"/".join(map(str, BURDENS))} = ' + ' / '.join(f'{pv.loc[(g, m), b]:.3f}' for b in BURDENS))
for k, v in HY.items(): Ls.append(f'{k}: M1 vs {v["b"]} on {v["set"]} ({v["events"]} events): {v["recall_a"]:.3f} vs {v["recall_b"]:.3f}, diff {v["diff"]:+.3f} CI [{v["ci_lo"]:+.3f}, {v["ci_hi"]:+.3f}] -> {"HOLDS" if v["holds"] else "does not hold"}')
Ls.append('M1 training: ' + ', '.join(f'{k} e{v["best_epoch"]}/{v["epochs_run"]}' for k, v in trainfo.items() if k.startswith('M1')) + ' | B6 ' + str(B6INFO.get('best_iteration')))
fl = DET[('M1', 'shift', PRIMARY)].query('pattern=="flicker"')['detected'].mean(); Ls.append(f'M1 flicker (held-out pattern) recall @{PRIMARY} = {fl:.3f}')
Ls += ['failed tests: ' + (', '.join(r[0] for r in ALL if not r[2]) or 'none'), '==== END ====']
print('\n'.join(Ls)); sb = OUT / 'reports' / 'f2_share_block.txt'
if not sb.exists(): sb.write_text('\n'.join(Ls))
for (m, g, b), df in DET.items(): df.to_csv(OUT / 'reports' / f'detections_{m}_{g}_b{b}.csv', index=False)

==== F2 SHARE BLOCK (copy everything between the two ==== lines) ====
F2 COMPLETE | tests passed 34/34 | test days opened: NO | Tesla T4 | libs 85e7c1 54b345 a67aa3 375cb1 776e71 324ca6
std   M1       recall @20/100/400 = 0.494 / 0.572 / 0.675
std   M0       recall @20/100/400 = 0.433 / 0.514 / 0.611
std   B3d      recall @20/100/400 = 0.319 / 0.364 / 0.467
std   B6       recall @20/100/400 = 0.381 / 0.444 / 0.525
std   B1-top5  recall @20/100/400 = 0.003 / 0.019 / 0.044
std   B1-all   recall @20/100/400 = 0.000 / 0.003 / 0.042
std   B5       recall @20/100/400 = 0.003 / 0.069 / 0.333
shift M1       recall @20/100/400 = 0.400 / 0.531 / 0.694
shift M0       recall @20/100/400 = 0.358 / 0.431 / 0.550
shift B3d      recall @20/100/400 = 0.236 / 0.294 / 0.433
shift B6       recall @20/100/400 = 0.297 / 0.369 / 0.508
shift B1-top5  recall @20/100/400 = 0.011 / 0.025 / 0.086
shift B1-all   recall @20/100/400 = 0.008 / 0.019 / 0.072
shift B5       recall @20/100/400 = 0.003 / 0.064 / 0.361
H3